[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u3_sistemas.ipynb)

# Álg. U3 · Sistemas lineales y rango
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 3 del bloque de álgebra lineal. Tiene una sección por subtema, en el mismo orden que el libro:

1. Eliminación de Gauss y Gauss-Jordan
2. Regla de Cramer
3. Ecuaciones vectorial y matricial; rango
4. Clasificación y sistemas homogéneos
5. Interpretación geométrica en 2D y 3D

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría, código que la recorre paso a paso, una calculadora, sus casos de prueba y una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U3` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u3_sistemas/NN_*.py`; los fragmentos que el libro imprime, de `alg/u3_sistemas/fragmentos/`.

**Convenciones.**

- Punto decimal. Un sistema se escribe como su **matriz aumentada** $[A\,|\,\mathbf b]$, **por renglones**: cada renglón es una ecuación, los renglones se separan con `;` o con un salto de línea, y las entradas con comas o espacios. **La última columna es $\mathbf b$.** Por ejemplo, `1, 1, 6; 2, -1, 3` significa $x+y=6$, $2x-y=3$. También se acepta la forma de Python `[[1, 1, 6], [2, -1, 3]]` y una barra antes de $\mathbf b$: `1, 1 | 6; 2, -1 | 3`.
- Puedes escribir fracciones como `2/3`; los decimales se convierten a fracciones exactas (`0.1` se lee como $\tfrac{1}{10}$), así que las cuentas internas no acumulan error de redondeo. Se acepta el signo menos tipográfico (−).
- Las incógnitas se llaman $x, y, z$ cuando son tres o menos, y $x_1,\dots,x_6$ cuando son más. Las calculadoras aceptan hasta 6 ecuaciones y 6 incógnitas: una matriz aumentada de $6\times 7$ a lo más.
- Los renglones se nombran $R_1, R_2,\dots$ y las columnas se cuentan desde 1. Las operaciones de renglón se escriben `R2 ← R2 - 2·R1` (al renglón 2 le restas 2 veces el renglón 1), `R2 ↔ R3` (intercambias los renglones 2 y 3) y `R1 ← (1/2)·R1` (multiplicas el renglón 1 por 1/2).
- En lo que muestran las calculadoras, el signo menos es siempre `-`, también en las ecuaciones y en las operaciones de renglón.
- Los decimales que se muestran se *redondean* (no se truncan) a las cifras significativas que elijas.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re, io, textwrap, contextlib
from decimal import Decimal, ROUND_HALF_UP, localcontext
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección "3d")
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook: leer matrices exactas, redondear y mostrar
_NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")
_AYUDA_NUMERO = "usa enteros, fracciones como 2/3 o decimales con punto"


_LIMITE = Fraction(10) ** 15                  # entradas entre 1e-15 y 1e15 en valor absoluto (o 0)


def _en_rango(q, texto):
    if q != 0 and not Fraction(1) / _LIMITE <= abs(q) <= _LIMITE:
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    return sp.Rational(q.numerator, q.denominator)


def numero(valor, nombre="el número"):
    """Un número como racional exacto de sympy: '0.1' -> 1/10, '2/3' -> 2/3, 5 -> 5, 1.2 -> 6/5."""
    if valor is None:
        raise ValueError(f"{nombre} está vacío: escribe un número.")
    if isinstance(valor, bool):
        raise ValueError(f"«{valor}» no es un número: {_AYUDA_NUMERO}.")
    if isinstance(valor, (int, np.integer)):
        return _en_rango(Fraction(int(valor)), valor)
    if isinstance(valor, (float, np.floating)) or (isinstance(valor, sp.Basic) and valor.is_Float):
        v = float(valor)
        if not math.isfinite(v):
            raise ValueError(f"{nombre} no es un número finito.")
        return _en_rango(Fraction(repr(v)), repr(v))      # repr da el decimal más corto: 0.1 -> '0.1' -> 1/10 exacto
    if isinstance(valor, sp.Basic):
        if valor.is_Rational:
            return valor                             # ya es exacto (por ejemplo, un resultado intermedio): no se acota
        raise ValueError(f"«{valor}» no es un número racional: {_AYUDA_NUMERO}.")
    texto = re.sub(r"\s+", "", str(valor)).replace("−", "-")      # acepta el signo menos tipográfico
    partes = texto.split("/")
    formas = [_NUMERO.fullmatch(p) for p in partes]
    if not texto or len(partes) > 2 or not all(formas):
        raise ValueError(f"«{str(valor).strip()}» no es un número: {_AYUDA_NUMERO}." if texto else
                         f"{nombre} está vacío: escribe un número.")
    if any(f.group(2) and abs(int(f.group(2)[1:])) > 30 for f in formas):   # 1e999999 no se calcula: se rechaza antes
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    num, *den = (Fraction(p) for p in partes)
    if den and den[0] == 0:
        raise ValueError(f"«{texto}» divide entre cero.")
    return _en_rango(num / den[0] if den else num, texto)


def _entradas(k):
    return f"{k} entrada" if k == 1 else f"{k} entradas"


def _desde_renglones(renglones, nombre):
    """Lista de renglones (cada uno, lista de entradas) -> Matrix de racionales, con mensajes claros."""
    if not renglones or not renglones[0]:
        raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 2; 3, 4")
    n = len(renglones[0])
    salida = []
    for i, renglon in enumerate(renglones, 1):
        if len(renglon) != n:
            raise ValueError(f"en {nombre}, el renglón {i} tiene {_entradas(len(renglon))} y el 1 tiene {n}: "
                             "todos los renglones deben tener el mismo número de entradas")
        fila = []
        for j, v in enumerate(renglon, 1):
            try:
                fila.append(numero(v))
            except ValueError as err:
                raise ValueError(f"en {nombre}, renglón {i}, columna {j}: {err}") from None
        salida.append(fila)
    return sp.Matrix(salida)


def matriz(texto, nombre="la matriz"):
    """Lee una matriz escrita por renglones ('1, 2; 3, 4' o un renglón por línea) como Matrix de racionales exactos."""
    if not isinstance(texto, str):
        return como_matriz(texto, nombre)
    t = texto.replace("−", "-")
    t = re.sub(r"[\]\)]\s*,?\s*[\[\(]", ";", t)            # [[1, 2], [3, 4]] o (1, 2), (3, 4): cada grupo es un renglón
    t = re.sub(r"[\[\]\(\)]", " ", t)
    renglones = [r.strip() for r in re.split(r"[;\n]", t)]
    renglones = [r for r in renglones if r]
    entradas = []
    for i, r in enumerate(renglones, 1):
        partes = re.split(r"\s*,\s*|\s+", r)
        if "" in partes:
            raise ValueError(f"en {nombre}, el renglón {i} tiene una entrada vacía: "
                             "revisa que no haya dos comas seguidas ni una coma al principio o al final")
        entradas.append(partes)
    try:
        return _desde_renglones(entradas, nombre)
    except ValueError as err:
        if "todos los renglones deben tener" in str(err):
            raise ValueError(f"{err}{pista_coma(texto)}") from None
        raise


def pista_coma(texto):
    """Si el texto tiene algo como 1,5 (coma entre dígitos, sin espacio), la pista de usar punto decimal."""
    return "; si escribiste decimales, usa punto decimal: 1.5, no 1,5" if isinstance(texto, str) and re.search(r"\d,\d", texto) else ""


def como_matriz(A, nombre="la matriz"):
    """Acepta texto, Matrix de sympy, lista de listas o arreglo de numpy y devuelve una Matrix de racionales exactos."""
    if isinstance(A, str):
        return matriz(A, nombre)
    if isinstance(A, sp.MatrixBase):
        renglones = A.tolist()
    elif isinstance(A, np.ndarray):
        if A.ndim > 2:
            raise ValueError(f"{nombre} debe ser una tabla de números (renglones y columnas)")
        renglones = np.atleast_2d(A).tolist()
    elif isinstance(A, (list, tuple)):
        anidada = [isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)) for f in A]
        if A and all(anidada):
            renglones = [list(f) for f in A]
        elif not any(anidada):
            renglones = [list(A)]                  # una lista sola es un renglón
        else:
            raise ValueError(f"en {nombre}, mezclaste números sueltos con renglones")
    else:
        renglones = [[A]]
    return _desde_renglones(renglones, nombre)


def vector(texto, nombre="el vector"):
    """Lee un vector ('1, 2, 3', '1; 2; 3', (1, 2, 3) o [1, 2, 3]) como columna de racionales exactos."""
    try:
        v = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):          # vacío: el ejemplo debe ser de vector, no de matriz
            raise ValueError(f"escribe {nombre} como lista de números, por ejemplo: 1, 2, 3") from None
        raise
    if v.rows == 1 or v.cols == 1:
        return v.reshape(v.rows * v.cols, 1)
    raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 1, 2, 3 (escribiste una matriz {dims(v)})")


def dims(M):
    """Tamaño 'm×n' de una matriz."""
    return f"{M.rows}×{M.cols}"


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Un racional exacto se redondea desde su valor exacto (p/q con 50 cifras), no desde su aproximación en punto
    flotante: 1249999999999999999/10^19 con 2 cifras da 0.12, no 0.13."""
    with localcontext() as ctx:
        ctx.prec = 50
        if isinstance(valor, (sp.Rational, Fraction, int, np.integer)) and not isinstance(valor, bool):
            p, q = (int(valor.p), int(valor.q)) if isinstance(valor, sp.Rational) else (
                (valor.numerator, valor.denominator) if isinstance(valor, Fraction) else (int(valor), 1))
            d = Decimal(p) / Decimal(q)
        else:
            v = float(valor)
            if math.isnan(v):
                return "no definido"
            if math.isinf(v):
                return "+∞" if v > 0 else "-∞"
            d = Decimal(repr(v))
        if d == 0:
            return "0"
        q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        if q.adjusted() != d.adjusted():      # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
            q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        e = q.adjusted()
        if e > 15 or e < -4:                  # muy grandes o muy chicos: notación científica, 1.234×10^-7
            return f"{format(q.scaleb(-e), 'f')}×10^{e}"
        if q.adjusted() >= n:                 # enteros grandes: sin notación científica
            return f"{int(q):d}"
        return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def _mensaje(fn):
    """Texto del ValueError que lanza fn() ('' si no lanza): para probar que el mensaje explica el error."""
    try:
        fn()
    except ValueError as err:
        return str(err)
    return ""


# --- formato de números y matrices ---------------------------------------------------------------
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


def _sub(*indices):
    """Subíndices: _sub(1, 2) -> '₁₂'; con índices de dos cifras los separa con coma."""
    if all(k < 10 for k in indices):
        return "".join(str(k) for k in indices).translate(_SUB)
    return "₍" + ",".join(str(k) for k in indices).translate(_SUB) + "₎"


def _paren(v):
    """Un número listo para ir en un producto: negativos y fracciones entre paréntesis."""
    v = sp.sympify(v)
    return str(v) if (v >= 0 and v.is_integer) else f"({v})"


def _en_linea(M):
    """Matriz en una línea con la misma notación de entrada: [2, 3; 5, -2]."""
    return "[" + "; ".join(", ".join(str(e) for e in M.row(i)) for i in range(M.rows)) + "]"


def _tupla(v, n=None):
    """Vector como (a, b, c); con n, además sus decimales redondeados a n cifras."""
    exacto = "(" + ", ".join(str(e) for e in v) + ")"
    if n is None or all(e.is_integer for e in v):
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")"


def _aprox(v, n):
    """Decimales redondeados de un vector: '(0.5000, 1.333)'; con una sola entrada, sin paréntesis."""
    textos = [cifras(e, n) for e in v]
    return textos[0] if len(textos) == 1 else "(" + ", ".join(textos) + ")"


def _plano(M, fmt=str, corte=None):
    """Texto plano de una matriz con columnas alineadas; corte = columna tras la que se dibuja una barra."""
    celdas = [[fmt(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]
    anchos = [max(len(celdas[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    lineas = []
    for fila in celdas:
        partes = [c.rjust(a) for c, a in zip(fila, anchos)]
        if corte:
            partes = partes[:corte] + ["|"] + partes[corte:]
        lineas.append("[ " + "  ".join(partes) + " ]")
    return "\n".join(lineas)


_TEX = [("⁻¹", "^{-1}"), ("ᵀ", "^{T}"), ("·", r"\cdot "), ("×", r"\times "), ("≈", r"\approx "),
        ("−", "-"), ("det ", r"\det "), ("|", r"\,|\,")]
_SUB_TEX = re.compile("[₀₁₂₃₄₅₆₇₈₉]+")


_PALABRAS = re.compile(r"(?<![\\A-Za-zÁÉÍÓÚáéíóúñ])([A-Za-zÁÉÍÓÚáéíóúñ]{3,}(?:\s+[A-Za-zÁÉÍÓÚáéíóúñ]{2,})*)")


def _a_tex(nombre):
    """Nombre de una cantidad en LaTeX: 'det A₁' -> '\\det A_{1}'; las palabras ('forma escalonada de') van en \\text."""
    for a, b in _TEX:
        nombre = nombre.replace(a, b)
    nombre = _SUB_TEX.sub(lambda m: "_{" + m.group(0).translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")) + "}", nombre)
    return _PALABRAS.sub(lambda m: r"\text{" + m.group(1) + "} ", nombre)


def _tex_decimales(M, n):
    filas = r" \\ ".join(" & ".join(cifras(M[i, j], n) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{matrix}" + filas + r"\end{matrix}\right]"


class _Formula(Math):
    """Math de IPython con una versión en texto plano (la que se ve donde no se dibuja LaTeX)."""
    def __init__(self, tex, texto):
        super().__init__(tex)
        self._texto = texto

    def __repr__(self):
        return self._texto


class _Bloque:
    """Salida en Markdown (texto en recuadro y fórmulas en LaTeX) con su versión en texto plano."""
    def __init__(self, partes):
        self._partes = partes

    @staticmethod
    def _envuelve(texto, ancho=100):
        """Corta los renglones largos (en un recuadro de código no se ajustan solos); conserva la sangría."""
        lineas = []
        for linea in texto.rstrip("\n").split("\n"):
            sangria = " " * (len(linea) - len(linea.lstrip(" ")))
            lineas += textwrap.wrap(linea, ancho, subsequent_indent=sangria + "  ") if len(linea) > ancho else [linea]
        return "\n".join(lineas)

    def _repr_markdown_(self):
        return "\n\n".join(f"$${tex}$$" if tipo == "tex" else "```text\n" + self._envuelve(texto) + "\n```"
                            for tipo, tex, texto in self._partes)

    def __repr__(self):
        return "\n".join(texto.rstrip("\n") for _, _, texto in self._partes)


_BLOQUES = []      # bloques abiertos: lo que se muestra dentro de uno se junta y se entrega de una vez


@contextlib.contextmanager
def bloque():
    """Junta lo que se imprime y las fórmulas que se muestran dentro del bloque y lo entrega como UNA sola salida.
    Las calculadoras lo usan para que su resultado aparezca completo de una vez, en el orden en que se produjo."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue()))
            buf.seek(0); buf.truncate()

    _BLOQUES.append((partes, vacia))
    try:
        with contextlib.redirect_stdout(buf):
            yield
    finally:
        vacia()
        _BLOQUES.pop()
        if partes:
            _emite(_Bloque(partes)) if _BLOQUES else display(_Bloque(partes))


def _emite(formula):
    """Muestra una fórmula, o la agrega al bloque abierto si lo hay."""
    if not _BLOQUES:
        display(formula)
        return
    partes, vacia = _BLOQUES[-1]
    vacia()
    if isinstance(formula, _Bloque):
        partes.extend(formula._partes)
    else:
        partes.append(("tex", formula.data, formula._texto))


def muestra(nombre, M, n=None):
    """Muestra 'nombre = M' con LaTeX (con su tamaño si es matriz). Con n agrega los decimales redondeados a n cifras."""
    if isinstance(M, sp.MatrixBase):
        tex = f"{_a_tex(nombre)} = {sp.latex(M)}"
        texto = f"{nombre}  ({dims(M)}) =\n{_plano(M)}"
        if n is not None and not all(e.is_integer for e in M):
            tex += r" \approx " + _tex_decimales(M, n)
            texto += f"\n≈\n{_plano(M, lambda e: cifras(e, n))}"
        tex += rf"\qquad ({M.rows}\times {M.cols})"
    else:
        M = sp.sympify(M)
        tex, texto = f"{_a_tex(nombre)} = {sp.latex(M)}", f"{nombre} = {M}"
        if n is not None and M.is_number and not M.is_integer:
            tex += rf" \approx {cifras(M, n)}"
            texto += f" ≈ {cifras(M, n)}"
    _emite(_Formula(tex, texto))


def compara(mia, ref, pista=""):
    """'coinciden' o en qué entradas difiere tu matriz de la referencia, más una pista."""
    if mia.shape != ref.shape:
        return f"NO coinciden: tu matriz es {dims(mia)} y la de referencia es {dims(ref)}. {pista}".strip()
    malas = [f"({i + 1}, {j + 1})" for i in range(ref.rows) for j in range(ref.cols) if mia[i, j] != ref[i, j]]
    if not malas:
        return "coinciden"
    cuales = "la entrada" if len(malas) == 1 else "las entradas"
    return f"NO coinciden en {cuales} {', '.join(malas)} (renglón, columna). {pista}".strip()

In [ ]:
# Utilidades nuevas de esta unidad: sistemas como matriz aumentada, eliminación exacta y solución general
_MAX_ECUACIONES, _MAX_INCOGNITAS = 6, 6       # matriz aumentada de 6×7 a lo más


def _ecuaciones(k):
    return f"{k} ecuación" if k == 1 else f"{k} ecuaciones"


def _incognitas(k):
    return f"{k} incógnita" if k == 1 else f"{k} incógnitas"


def _revisa_tamano(m, n, nombre, cosa="incógnitas"):
    """Rechaza más de 6 renglones o más de 6 incógnitas (columnas de A)."""
    limite = "una matriz aumentada de 6×7 a lo más" if cosa == "incógnitas" else "matrices de hasta 6×6"
    if m > _MAX_ECUACIONES:
        raise ValueError(f"{nombre} tiene {m} renglones; esta calculadora acepta hasta {_MAX_ECUACIONES} ({limite})")
    if n > _MAX_INCOGNITAS:
        if cosa == "incógnitas":
            raise ValueError(f"{nombre} tiene {n + 1} columnas, es decir, {n} incógnitas más la columna b; esta calculadora "
                             f"acepta hasta {_MAX_INCOGNITAS} incógnitas ({limite})")
        raise ValueError(f"{nombre} tiene {n} columnas; esta calculadora acepta hasta {_MAX_INCOGNITAS} ({limite})")


def aumentada(texto, nombre="la matriz aumentada"):
    """Lee un sistema escrito como su matriz aumentada [A | b] por renglones; la última columna es b.
    '1, 1, 6; 2, -1, 3' es x + y = 6, 2x - y = 3. Acepta una barra antes de b: '1, 1 | 6; 2, -1 | 3'."""
    if isinstance(texto, str):
        texto = texto.replace("|", ",")
    try:
        M = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):          # vacío: el ejemplo debe ser de un sistema
            raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 1, 6; 2, -1, 3 "
                             "(cada renglón es una ecuación y la última columna es b)") from None
        raise
    if M.cols < 2:
        raise ValueError(f"{nombre} necesita al menos dos columnas: los coeficientes y, al final, la columna b "
                         "(por ejemplo, «2, 6» es la ecuación 2x = 6)")
    try:
        _revisa_tamano(M.rows, M.cols - 1, nombre)
    except ValueError as err:
        raise ValueError(f"{err}{pista_coma(texto)}") from None
    return M


def matriz_limitada(texto, nombre="A"):
    """Una matriz (no aumentada) de hasta 6×6, con los mismos mensajes que matriz()."""
    M = como_matriz(texto, nombre)
    try:
        _revisa_tamano(M.rows, M.cols, nombre, cosa="columnas")
    except ValueError as err:
        raise ValueError(f"{err}{pista_coma(texto)}") from None
    return M


def de_columnas(*columnas):
    """Matriz cuyas columnas son los vectores dados: de_columnas('1, 0, 1', '0, 1, 1')."""
    vs = [vector(c, f"la columna {j}") for j, c in enumerate(columnas, 1)]
    if not vs:
        raise ValueError("escribe al menos una columna")
    if len({v.rows for v in vs}) > 1:
        raise ValueError("todas las columnas deben tener el mismo número de entradas")
    return sp.Matrix.hstack(*vs)


def nombres_incognitas(n):
    """x, y, z con tres incógnitas o menos; x₁, ..., xₙ con más."""
    return ["x", "y", "z"][:n] if n <= 3 else [f"x{_sub(k)}" for k in range(1, n + 1)]


def parametros(k):
    """Nombres de k parámetros, en el orden de las variables libres: s, t, u, v, w (con más de cinco: s₁, s₂, ...)."""
    if k == 0:
        return []
    if k <= 5:
        return list(sp.symbols("s t u v w")[:k])
    return list(sp.symbols(" ".join(f"s{_sub(i)}" for i in range(1, k + 1))))


def _factor(a):
    """Coeficiente delante de un renglón: 1 -> '', 2 -> '2·', 1/2 -> '(1/2)·', -1 -> '(-1)·'."""
    if a == 1:
        return ""
    if a.is_integer and a > 0:
        return f"{a}·"
    return f"({a})·"


def _op_suma(i, m, k):
    """Texto de R_i ← R_i - m·R_k con el signo que toca: m = 2 -> 'R2 ← R2 - 2·R1'; m = -3 -> 'R2 ← R2 + 3·R1'."""
    signo, a = ("-", m) if m > 0 else ("+", -m)
    return f"R{i} ← R{i} {signo} {_factor(a)}R{k}"


def _elimina(M, n_a, con_b=True):
    """Eliminación de Gauss y después de Jordan, exacta, sobre una copia de M (Matrix de racionales).

    Los pivotes se buscan en las primeras n_a columnas (las de A); se intercambian renglones solo si el pivote es 0
    y abajo hay una entrada distinta de cero (como a mano). Con con_b, al final se busca un pivote en la columna n_a
    (la de b): si aparece, hay un renglón 0 = c y la forma escalonada lo es también de [A | b].
    La fase de Jordan deja cada pivote en 1 y ceros arriba de él, también el de la columna de b si lo hay: en un sistema
    incompatible, el renglón 0 = c queda como 0 = 1 y la columna de b se limpia hacia arriba (forma reducida de verdad).
    'imposible' guarda el renglón y el valor c de la forma escalonada, antes de normalizar.
    Devuelve un diccionario: E (escalonada), R (reducida), pivotes (columnas de A con pivote, desde 0), libres
    (columnas de A sin pivote), r (rango de A), r_aum (rango de [A | b]), imposible ((renglón, c) o None) y pasos,
    una lista de (tipo, texto, matriz después de la operación o None) con tipo 'fase', 'nota' u 'op'."""
    E = M.copy()
    m = E.rows
    pasos = [("fase", "Gauss: ceros debajo de cada pivote (forma escalonada)", None)]
    pivotes, libres, f, imposible = [], [], 0, None
    for c in range(n_a + (1 if con_b and M.cols > n_a else 0)):
        es_b = c == n_a
        k = next((i for i in range(f, m) if E[i, c] != 0), None)
        if k is None:
            if not es_b:
                libres.append(c)
                motivo = "ya no quedan renglones abajo" if f >= m else f"de R{f + 1} hacia abajo todo es cero"
                pasos.append(("nota", f"la columna {c + 1} no tiene pivote: {motivo}", None))
            continue
        if k != f:
            pasos.append(("nota", f"el pivote de la columna {c + 1} es cero; se intercambian R{f + 1} y R{k + 1}", None))
            E.row_swap(f, k)
            pasos.append(("op", f"R{f + 1} ↔ R{k + 1}", E.copy()))
        for i in range(f + 1, m):
            if E[i, c] != 0:
                mult = E[i, c] / E[f, c]
                E[i, :] = E[i, :] - mult * E[f, :]
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), E.copy()))
        if es_b:
            imposible = (f + 1, E[f, c])
            pasos.append(("nota", f"R{f + 1} quedó como 0 = {E[f, c]}: ningún valor de las incógnitas lo cumple", None))
        else:
            pivotes.append(c)
        f += 1
    if len(pasos) == 1:
        pasos.append(("nota", "la matriz ya estaba escalonada: no hace falta ninguna operación", None))
    R = E.copy()
    pasos.append(("fase", "Jordan: cada pivote en 1 y ceros también arriba (forma escalonada reducida)", None))
    hechas = len(pasos)
    todos = list(enumerate(pivotes)) + ([(len(pivotes), n_a)] if imposible is not None else [])
    for f, c in reversed(todos):
        if R[f, c] != 1:
            k = 1 / R[f, c]
            R[f, :] = k * R[f, :]
            pasos.append(("op", f"R{f + 1} ← {_factor(k)}R{f + 1}", R.copy()))
        for i in range(f):
            if R[i, c] != 0:
                mult = R[i, c]
                R[i, :] = R[i, :] - mult * R[f, :]
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), R.copy()))
    if len(pasos) == hechas:
        pasos.append(("nota", "la forma escalonada ya era reducida: no hace falta ninguna operación", None))
    return {"E": E, "R": R, "pivotes": pivotes, "libres": libres, "r": len(pivotes),
            "r_aum": len(pivotes) + (imposible is not None), "imposible": imposible, "pasos": pasos}


def operaciones(el):
    """Solo los textos de las operaciones de renglón de un resultado de _elimina."""
    return [texto for tipo, texto, _ in el["pasos"] if tipo == "op"]


def _imprime_pasos(pasos, corte=None, sangria="  "):
    for tipo, texto, M in pasos:
        if tipo == "fase":
            print(f"— {texto} —")
        elif tipo == "nota":
            print(f"{sangria}Nota: {texto}.")
        else:
            print(f"{sangria}{texto}")
            print("\n".join(sangria + "  " + linea for linea in _plano(M, corte=corte).split("\n")))


def solucion_general(R, pivotes, n):
    """Solución de un sistema compatible a partir de su forma reducida R (con la columna b al final).
    Devuelve (x, particular, direcciones, parámetros): cada variable libre es un parámetro (s, t, u, ... en el orden
    de las libres), x = particular + s·d₁ + t·d₂ + ..., y la particular es la que tiene las libres en 0."""
    libres = [c for c in range(n) if c not in pivotes]
    ps = parametros(len(libres))
    particular = sp.zeros(n, 1)
    for f, c in enumerate(pivotes):
        particular[c] = R[f, n]
    direcciones = []
    for c_libre in libres:
        d = sp.zeros(n, 1)
        d[c_libre] = 1
        for f, c in enumerate(pivotes):
            d[c] = -R[f, c_libre]
        direcciones.append(d)
    x = particular + sum((p * d for p, d in zip(ps, direcciones)), sp.zeros(n, 1))
    return x, particular, direcciones, ps


def _coeficiente(a, letra, primero):
    """Un término 'a·letra' con el signo que toca: (-2, 's', False) -> '- 2s'; (1/4, 's', True) -> '(1/4)s'."""
    mag = abs(a)
    coef = "" if (mag == 1 and letra) else (f"{mag}" if mag.is_integer else f"({mag})")
    if primero:
        return f"{'-' if a < 0 else ''}{coef}{letra}"
    return f"{'-' if a < 0 else '+'} {coef}{letra}"


def _lineal(e, ps):
    """Expresión lineal en los parámetros, con la constante primero: '4 - 2s', '(1/4)s', '7 - 2s - 3t'."""
    e = sp.expand(sp.sympify(e))
    constante = e.subs({p: 0 for p in ps})
    partes = [str(constante)] if constante != 0 else []
    for p in ps:
        a = e.coeff(p)
        if a != 0:
            partes.append(_coeficiente(a, str(p), not partes))
    return " ".join(partes) if partes else "0"


def _ecuacion(fila, nombres):
    """Renglón [a₁, ..., aₙ, b] como ecuación: '2x - y + z = 3'; si todos los coeficientes son 0, '0 = b'."""
    terminos = []
    for a, v in zip(list(fila)[:-1], nombres):
        if a != 0:
            terminos.append(_coeficiente(a, v, not terminos))
    return f"{' '.join(terminos) if terminos else '0'} = {list(fila)[-1]}"


def _imprime_sistema(Ab, sangria="  "):
    m, n = Ab.rows, Ab.cols - 1
    print(f"Sistema de {_ecuaciones(m)} con {_incognitas(n)}:")
    for i in range(m):
        print(f"{sangria}{_ecuacion(Ab.row(i), nombres_incognitas(n))}")


def _tex_aumentada(M, corte):
    columnas = "c" * corte + "|" + "c" * (M.cols - corte)
    filas = r" \\ ".join(" & ".join(sp.latex(M[i, j]) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{array}{" + columnas + "}" + filas + r"\end{array}\right]"


def muestra_aumentada(nombre, M, corte):
    """Muestra una matriz aumentada con la barra después de la columna 'corte'."""
    _emite(_Formula(f"{_a_tex(nombre)} = {_tex_aumentada(M, corte)}", f"{nombre} =\n{_plano(M, corte=corte)}"))


def _lista(cosas):
    """Números en una frase: [1, 2, 3] -> '1, 2 y 3'. (Para nombres de incógnitas usa _coma: 'x y z' confundiría.)"""
    cosas = [str(c) for c in cosas]
    return cosas[0] if len(cosas) == 1 else ", ".join(cosas[:-1]) + " y " + cosas[-1]


def _coma(cosas):
    """Nombres de incógnitas o parámetros separados con comas: ['x', 'y'] -> 'x, y'."""
    return ", ".join(str(c) for c in cosas)


def _flotante(M):
    """Matrix de sympy -> arreglo de numpy en punto flotante."""
    return np.array(M.tolist(), dtype=float)


# Gráficas: un color y un estilo de línea por ecuación, en orden fijo, para que se lean también en blanco y negro
_COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
_ESTILOS = ["-", "--", ":", "-.", (0, (5, 1, 1, 1, 1, 1)), (0, (1, 4))]          # continuo, discontinuo, punteado...


_LLAMADAS_SOLVE = [0]      # cuántas veces se llamó a numpy.linalg.solve (las pruebas revisan que no se llame de más)


_AVISO_SINGULAR = ("numpy.linalg.solve no pudo resolverlo en punto flotante (la matriz es singular para la máquina), "
                   "aunque la solución exacta existe: el sistema está muy mal condicionado")


def _numpy_solve(A, b):
    """(x, None) con numpy.linalg.solve en punto flotante, solo para comparar con la solución exacta; (None, aviso)
    si numpy no puede: una matriz invertible en fracciones puede ser singular para la máquina."""
    _LLAMADAS_SOLVE[0] += 1
    try:
        with np.errstate(all="ignore"):
            x = np.linalg.solve(_flotante(A), _flotante(b).ravel())
    except np.linalg.LinAlgError:
        return None, _AVISO_SINGULAR
    if not np.all(np.isfinite(x)):
        return None, _AVISO_SINGULAR
    return x, None


def normaliza_texto(texto):
    """Minúsculas, sin acentos ni signos de puntuación y con un solo espacio entre palabras: '¿Se cortan?' -> 'se cortan'."""
    import unicodedata
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


def compara_vector(mio, ref, nombres, pista=""):
    """'coinciden' o qué incógnitas difieren de la referencia, más una pista."""
    if mio.rows != ref.rows:
        return (f"NO coinciden: escribiste {_entradas(mio.rows).replace('entrada', 'número')} y el sistema tiene "
                f"{_incognitas(ref.rows)}. {pista}").strip()
    malas = [nombres[k] for k in range(ref.rows) if mio[k] != ref[k]]
    if not malas:
        return "coinciden"
    return f"NO coinciden en {_coma(malas)}. {pista}".strip()


def evalua(pruebas):
    """Ejecuta una vez cada caso (nombre, función que devuelve True o False). Devuelve [(nombre, ok)]; un caso
    que lanza una excepción cuenta como falla y su nombre dice por qué, sin interrumpir a los demás."""
    resultados = []
    for nombre, prueba in pruebas:
        try:
            ok = bool(prueba())
        except Exception as err:
            ok, nombre = False, f"{nombre}  (error: {type(err).__name__}: {err})"
        resultados.append((nombre, ok))
    return resultados


def corre_pruebas(pruebas, seccion):
    """Escribe OK o FALLA en cada caso y el resumen de la sección."""
    resultados = evalua(pruebas)
    for nombre, ok in resultados:
        print("OK  " if ok else "FALLA", nombre)
    print(f"Sección {seccion}: {sum(ok for _, ok in resultados)}/{len(pruebas)} casos pasan")

## 1. Eliminación de Gauss y Gauss-Jordan

Un sistema de $m$ ecuaciones con $n$ incógnitas se guarda en su **matriz aumentada** $[A\,|\,\mathbf b]$: cada renglón es una ecuación, las primeras $n$ columnas son los coeficientes y la última es $\mathbf b$. Por ejemplo,

$$\begin{aligned}2x+3y&=8\\ x-y&=-1\end{aligned}\qquad\longleftrightarrow\qquad\left[\begin{array}{cc|c}2&3&8\\1&-1&-1\end{array}\right].$$

**Operaciones elementales de renglón.** No cambian las soluciones, porque cada una se deshace con otra del mismo tipo:

- intercambiar dos renglones: $R_i\leftrightarrow R_j$;
- multiplicar un renglón por un número $k\neq 0$: $R_i\leftarrow k\,R_i$;
- sumar a un renglón un múltiplo de **otro**: $R_i\leftarrow R_i-k\,R_j$, con $i\neq j$.

Cada operación actúa sobre el renglón completo, **incluida la columna de $\mathbf b$**.

**Formas escalonadas.** Una matriz está en *forma escalonada* si los renglones de ceros van al final y el **pivote** de cada renglón (su primera entrada distinta de cero) queda a la derecha del pivote del renglón de arriba. Está en *forma escalonada reducida* si, además, cada pivote vale 1 y es lo único distinto de cero en su columna.

- **Gauss** baja hasta la forma escalonada: en la columna del pivote $a_{kk}$, a cada renglón $i$ de abajo le restas $m_{ik}=a_{ik}/a_{kk}$ veces el renglón $k$, $R_i\leftarrow R_i-m_{ik}R_k$. Después despejas de la última ecuación a la primera (sustitución hacia atrás).
- **Gauss-Jordan** sigue hasta la forma reducida: cada pivote en 1 y ceros también arriba de él. Si la solución es única, queda escrita en la última columna.

Tres situaciones cambian el camino:

1. **Pivote cero.** Si en la posición del pivote hay un 0 y abajo hay una entrada distinta de cero, intercambias esos renglones.
2. **Columna sin pivote.** Si de ese renglón hacia abajo toda la columna es cero, esa columna no tiene pivote y pasas a la siguiente. El sistema ya no tiene solución única; la sección 4 dice si tiene infinitas o ninguna.
3. **Renglón $(0\ \cdots\ 0\,|\,c)$ con $c\neq 0$.** Dice $0=c$, que nadie cumple: el sistema no tiene solución.

**Inversa por Gauss-Jordan.** Si $A$ es $n\times n$, Gauss-Jordan aplicado a $[A\,|\,I_n]$ termina en $[I_n\,|\,A^{-1}]$. Si alguna columna de $A$ se queda sin pivote, $A$ no tiene inversa (su determinante es 0).

**Comprueba siempre en las ecuaciones originales.** Si un renglón se operó mal, la solución cumple la ecuación equivocada, y solo la sustitución en el sistema de partida lo detecta: el **residuo** $\mathbf b-A\mathbf x$ debe ser $\mathbf 0$.

La calculadora trabaja con fracciones exactas e intercambia renglones solo cuando el pivote es cero, como harías a mano. Al final compara con `numpy.linalg.solve`, que trabaja en punto flotante y además intercambia renglones para usar el pivote más grande.

In [ ]:
def _filas_independientes(A):
    """Índices (desde 0) de renglones de A que no son combinación de los anteriores."""
    elegidas = []
    for i in range(A.rows):
        if sp.Matrix.vstack(*[A.row(k) for k in elegidas + [i]]).rank() > len(elegidas):
            elegidas.append(i)
    return elegidas


def gauss_jordan(Ab, pasos=True, n_cifras=4):
    """Resuelve el sistema [A | b] por Gauss-Jordan con fracciones exactas.

    Devuelve un diccionario con la forma escalonada, la reducida, las operaciones de renglón (texto), las columnas con
    y sin pivote (desde 1), si la solución es única, la solución, el residuo b - A x y la solución de numpy.linalg.solve
    (solo si la solución es única; si numpy no puede, None y un aviso). Con pasos=True imprime cada operación."""
    Ab = aumentada(Ab)
    m, n = Ab.rows, Ab.cols - 1
    A, b = Ab[:, :n], Ab[:, n]
    nombres = nombres_incognitas(n)
    el = _elimina(Ab, n, con_b=True)
    unica = el["r"] == n and el["imposible"] is None
    res = {"escalonada": el["E"], "reducida": el["R"], "operaciones": operaciones(el), "pasos": el["pasos"],
           "pivotes": [c + 1 for c in el["pivotes"]], "sin_pivote": [c + 1 for c in el["libres"]],
           "imposible": el["imposible"], "unica": unica, "solucion": None, "residuo": None,
           "numpy": None, "aviso_numpy": None, "ecuaciones_numpy": None, "mensajes": []}
    if el["libres"]:
        cols = el["libres"]
        cuales = "la columna" if len(cols) == 1 else "las columnas"
        texto = f"{cuales} {_lista([c + 1 for c in cols])} no {'tiene' if len(cols) == 1 else 'tienen'} pivote"
        if el["imposible"] is None:
            libres = _coma([nombres[c] for c in cols])
            texto += (f" ({libres} {'es libre' if len(cols) == 1 else 'son libres'}): no hay solución única; "
                      "la clasificación (infinitas soluciones o ninguna) se hace en la sección 4")
        res["mensajes"].append(texto)
    if el["imposible"] is not None:
        f, c = el["imposible"]
        res["mensajes"].append(f"aparece el renglón 0 = {c} (R{f} de la forma escalonada): el sistema no tiene solución; "
                               "la clasificación se hace en la sección 4")
    if unica:
        x = el["R"][:n, n]
        res["solucion"] = x
        res["residuo"] = b - A * x
        filas = list(range(m)) if m == n else _filas_independientes(A)
        res["numpy"], res["aviso_numpy"] = _numpy_solve(A.extract(filas, list(range(n))), b.extract(filas, [0]))
        res["ecuaciones_numpy"] = [i + 1 for i in filas]
    if pasos:
        _informe_gauss(Ab, res, n_cifras, con_operaciones=True)
    return res


def _sustitucion(fila, x):
    """'2·1 + 3·2' para el renglón (2, 3) y x = (1, 2)."""
    return " + ".join(f"{_paren(a)}·{_paren(v)}" for a, v in zip(fila, x))


def _informe_gauss(Ab, res, n_cifras, con_operaciones=True):
    m, n = Ab.rows, Ab.cols - 1
    A, b = Ab[:, :n], Ab[:, n]
    nombres = nombres_incognitas(n)
    _imprime_sistema(Ab)
    muestra_aumentada("[A | b]", Ab, n)
    if con_operaciones:
        _imprime_pasos(res["pasos"], corte=n)
    muestra_aumentada("forma escalonada", res["escalonada"], n)
    muestra_aumentada("forma escalonada reducida", res["reducida"], n)
    print("Columnas con pivote:", _lista(res["pivotes"]) if res["pivotes"] else "ninguna")
    for texto in res["mensajes"]:
        print(texto[0].upper() + texto[1:] + ".")
    if not res["unica"]:
        print("No se llama a numpy.linalg.solve: el sistema no tiene solución única.")
        return
    x = res["solucion"]
    decimales = "" if all(e.is_integer for e in x) else " ≈ " + _aprox(x, n_cifras)
    print("Solución única: " + ", ".join(f"{v} = {e}" for v, e in zip(nombres, x)) + decimales)
    print("Comprobación en las ecuaciones originales (residuo = b - A x):")
    for i in range(m):
        print(f"  ecuación {i + 1}: {_sustitucion(A.row(i), x)} = {(A.row(i) * x)[0]};  "
              f"b{_sub(i + 1)} = {b[i]};  residuo {res['residuo'][i]}")
    print("Residuo nulo: la solución cumple todas las ecuaciones." if res["residuo"] == sp.zeros(m, 1)
          else "El residuo NO es cero: revisa las operaciones.")
    xs = res["numpy"]
    if m != n:
        print(f"numpy.linalg.solve solo acepta sistemas cuadrados: se le dan las ecuaciones "
              f"{_lista(res['ecuaciones_numpy'])}, que son independientes; las demás son combinación de ellas.")
    if xs is None:
        print("Aviso: " + res["aviso_numpy"] + ".")
        return
    diferencia = float(np.max(np.abs(xs - _flotante(x).ravel())))
    print("numpy.linalg.solve (punto flotante): (" + ", ".join(cifras(v, n_cifras) for v in xs) + ")", end="; ")
    if diferencia == 0:
        print("coincide con la solución exacta.")
    elif diferencia < 1e-9 * max(1.0, float(np.max(np.abs(xs)))):
        print(f"coincide con la exacta salvo {cifras(diferencia, 2)}, error de redondeo del punto flotante.")
    else:
        print(f"difiere de la exacta en {cifras(diferencia, 2)}: el sistema está mal condicionado.")


def inversa_gj(A, pasos=False, n_cifras=4):
    """A⁻¹ exacta por Gauss-Jordan sobre [A | I]; ValueError si A no es cuadrada o no tiene inversa."""
    A = matriz_limitada(A, "A")
    if A.rows != A.cols:
        raise ValueError(f"la inversa solo existe para matrices cuadradas (esta es {dims(A)})")
    n = A.rows
    M = A.row_join(sp.eye(n))
    el = _elimina(M, n, con_b=False)
    if pasos:
        muestra_aumentada("[A | I]", M, n)
        _imprime_pasos(el["pasos"], corte=n)
    if el["r"] < n:
        cols = el["libres"]
        cuales = f"la columna {cols[0] + 1}" if len(cols) == 1 else f"las columnas {_lista([c + 1 for c in cols])}"
        raise ValueError(f"A no tiene inversa: al aplicar Gauss-Jordan a [A | I], {cuales} de A se "
                         f"{'queda' if len(cols) == 1 else 'quedan'} sin pivote (rango {el['r']} < {n}), así que la parte "
                         "izquierda nunca llega a I (equivale a det A = 0)")
    Ainv = el["R"][:, n:]
    if pasos:
        muestra_aumentada("[I | A⁻¹]", el["R"], n)
        muestra("A⁻¹", Ainv, n_cifras)
        P = A * Ainv
        muestra("A·A⁻¹", P)
        print("Verificación: A·A⁻¹ = I" if P == sp.eye(n) else "Verificación FALLÓ: A·A⁻¹ ≠ I")
    return Ainv

**Recorrido paso a paso.** El sistema $x+y+z=6$, $2x+2y+3z=15$, $x+3y+2z=13$. Al limpiar la columna 1, el pivote de la columna 2 queda en cero y hay que intercambiar renglones. Lee cada operación y la matriz que deja; al final, la comprobación en las ecuaciones originales y la comparación con `numpy`.

In [ ]:
_recorrido = gauss_jordan("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13")

La misma eliminación da la inversa: Gauss-Jordan sobre $[A\,|\,I]$ con $A=\begin{bmatrix}1&2\\3&4\end{bmatrix}$. La inversa sale con fracciones exactas.

In [ ]:
inversa_gj("1, 2; 3, 4", pasos=True);

**Calculadora.** Escribe la matriz aumentada por renglones (la última columna es $\mathbf b$). Prueba también los sistemas de los casos de prueba de abajo, por ejemplo uno con pivote cero (`0, 1, 2; 1, 1, 3`) o uno sin solución (`1, 1, 2; 1, 1, 3`).

In [ ]:
def calculadora_gauss(Ab_txt, ver_pasos, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            Ab = aumentada(Ab_txt)
            res = gauss_jordan(Ab, pasos=False)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_gauss(Ab, res, n_cifras, con_operaciones=ver_pasos)


widgets.interact(calculadora_gauss,
    Ab_txt=widgets.Textarea(value="2, 1, -1, 8\n-3, -1, 2, -11\n-2, 1, 2, -3", description="[A | b] =",
                            continuous_update=False, layout=widgets.Layout(width="340px", height="120px")),
    ver_pasos=widgets.Checkbox(value=True, description="mostrar cada operación"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

**Otra calculadora de esta sección: inversa por Gauss-Jordan.** Escribe una matriz cuadrada $A$ (hasta $6\times 6$); la calculadora aplica Gauss-Jordan a $[A\,|\,I]$, muestra cada operación y comprueba que $A\,A^{-1}=I$. Si una columna de $A$ se queda sin pivote, te dice que $A$ no tiene inversa.

In [ ]:
def calculadora_inversa_gj(A_txt, n_cifras):
    with bloque():
        try:
            inversa_gj(A_txt, pasos=True, n_cifras=n_cifras)
        except ValueError as err:     # que A no tenga inversa es un resultado, no un error de captura
            print("Resultado:" if str(err).startswith("A no tiene inversa") else "Revisa la entrada:", err)


widgets.interact(calculadora_inversa_gj,
    A_txt=widgets.Textarea(value="1, 2\n3, 4", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="340px", height="90px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
def _sin_solve(fn):
    """(resultado de fn(), True si fn() no llamó a numpy.linalg.solve)."""
    antes = _LLAMADAS_SOLVE[0]
    r = fn()
    return r, _LLAMADAS_SOLVE[0] == antes


_GJ = lambda texto: gauss_jordan(texto, pasos=False)
PRUEBAS_1 = [
    ("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13 → (1, 2, 3), con el intercambio R2 ↔ R3 y residuo 0",
     lambda: (lambda r: r["solucion"] == vector("1, 2, 3") and "R2 ↔ R3" in r["operaciones"]
                        and r["residuo"] == sp.zeros(3, 1))(_GJ("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13"))),
    ("2, 3, 8; 1, -1, -1 → (1, 2), igual que numpy.linalg.solve",
     lambda: (lambda r: r["solucion"] == vector("1, 2") and np.allclose(r["numpy"], [1, 2]))(_GJ("2, 3, 8; 1, -1, -1"))),
    ("2, 1, -1, 8; -3, -1, 2, -11; -2, 1, 2, -3 → (2, 3, -1)",
     lambda: _GJ("2, 1, -1, 8; -3, -1, 2, -11; -2, 1, 2, -3")["solucion"] == vector("2, 3, -1")),
    ("0.5, 0.25, 1; 0.1, 0.2, 0.5 → (1, 2) exacto: todas las cuentas en fracciones",
     lambda: (lambda r: r["solucion"] == vector("1, 2") and all(e.is_Rational for e in r["escalonada"])
                        and r["residuo"] == sp.zeros(2, 1))(_GJ("0.5, 0.25, 1; 0.1, 0.2, 0.5"))),
    ("0, 1, 2; 1, 1, 3 → intercambio en la columna 1 (R1 ↔ R2) y (1, 2)",
     lambda: (lambda r: r["operaciones"][0] == "R1 ↔ R2" and r["solucion"] == vector("1, 2")
                        and any("el pivote de la columna 1 es cero" in t for _, t, _m in r["pasos"]))(_GJ("0, 1, 2; 1, 1, 3"))),
    ("1, 2, 3; 2, 4, 6 → sin solución única (columna 2 sin pivote) y sin llamar a numpy.linalg.solve",
     lambda: (lambda r, limpio: limpio and not r["unica"] and r["sin_pivote"] == [2] and r["numpy"] is None
                                and "sección 4" in r["mensajes"][0])(*_sin_solve(lambda: _GJ("1, 2, 3; 2, 4, 6")))),
    ("1, 1, 2; 1, 1, 3 → aparece el renglón 0 = 1: no hay solución",
     lambda: (lambda r: r["imposible"] == (2, 1) and r["solucion"] is None
                        and any("0 = 1" in t and "no tiene solución" in t for t in r["mensajes"]))(_GJ("1, 1, 2; 1, 1, 3"))),
    ("inversa_gj: [[2,1],[5,3]] → [[3,-1],[-5,2]]; [[1,2],[2,4]] → no tiene inversa",
     lambda: inversa_gj("2, 1; 5, 3") == matriz("3, -1; -5, 2")
             and _mensaje(lambda: inversa_gj("1, 2; 2, 4")).startswith("A no tiene inversa")),
]
corre_pruebas(PRUEBAS_1, 1)

**Contrasta (sección 1).** Resuelve a mano $x+2y=7$, $3x-y=7$ con la matriz aumentada: escalona, despeja y comprueba en las dos ecuaciones. Escribe tu solución como texto, primero $x$ y luego $y$.

In [ ]:
mi_solucion = None     # escribe tu solución como texto, primero x y luego y; por ejemplo: "4, -1"

if mi_solucion is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mia = vector(str(mi_solucion), "tu solución")
    except ValueError as err:
        print("Revisa tu solución:", err)
    else:
        if mia.rows != 2:          # respuesta incompleta: la referencia todavía no se muestra
            print(f"Tu solución debe tener dos números, primero x y luego y (escribiste {mia.rows}): "
                  "complétala y vuelve a ejecutar la celda.")
        else:
            ref = gauss_jordan("1, 2, 7; 3, -1, 7", pasos=False)["solucion"]
            print(f"La calculadora da: x = {ref[0]}, y = {ref[1]}")
            print(compara_vector(mia, ref, ["x", "y"], "Con R2 ← R2 - 3·R1 el segundo renglón queda solo con y; "
                                                        "despeja y, sustituye en la primera ecuación y comprueba en las dos."))
            print("\nEl procedimiento completo:")
            gauss_jordan("1, 2, 7; 3, -1, 7")

## 2. Regla de Cramer

Si $A$ es $n\times n$ y $\det A\neq 0$, el sistema $A\mathbf x=\mathbf b$ tiene una sola solución, y cada incógnita es un cociente de determinantes:

$$x_i=\frac{\det A_i}{\det A},\qquad i=1,\dots,n,$$

donde $A_i$ es la matriz $A$ con su **columna $i$ cambiada por $\mathbf b$**.

**Por qué funciona.** Si $\mathbf b=x_1\mathbf a_1+\dots+x_n\mathbf a_n$ (con $\mathbf a_j$ la columna $j$ de $A$), en $A_i$ puedes restar a la columna $i$ los múltiplos $x_j\mathbf a_j$ de las demás columnas sin cambiar el determinante. Queda $x_i\mathbf a_i$ en su lugar y, al sacar el factor $x_i$, $\det A_i=x_i\det A$.

**Si $\det A=0$, no se divide.** La regla no da solución; solo asegura que no hay solución única. Hay dos casos:

- Algún $\det A_i\neq 0$: **no hay solución**, porque cualquier solución obligaría a $\det A_i=x_i\cdot\det A=x_i\cdot 0=0$.
- Todos los $\det A_i$ valen 0: **Cramer no decide**. El sistema puede tener infinitas soluciones o ninguna; por ejemplo, tres planos paralelos distintos tienen todos los determinantes en cero y no se cortan. Se decide comparando rangos (sección 4).

**Costo.** Con cofactores y sin aprovechar ceros, un determinante $n\times n$ cuesta $M_n=n\,(M_{n-1}+1)$ multiplicaciones, con $M_2=2$. Cramer calcula $n+1$ determinantes y hace $n$ divisiones: $(n+1)M_n+n$ operaciones. La eliminación de Gauss, contando multiplicaciones y divisiones, hace $\frac{n^3+3n^2-n}{3}$:

| $n$ | Cramer con cofactores | Gauss |
|---:|---:|---:|
| 2 | 8 | 6 |
| 3 | 39 | 17 |
| 4 | 204 | 36 |
| 10 | 68 588 310 | 430 |

Cramer crece como $n!$ y Gauss como $n^3$. Úsalo en sistemas $2\times 2$ y $3\times 3$, o cuando quieras una fórmula con letras para una incógnita; para sistemas más grandes, usa eliminación.

In [ ]:
def cramer(Ab, pasos=True, n_cifras=4):
    """Regla de Cramer exacta para un sistema cuadrado [A | b].

    Devuelve un diccionario con det A, la lista de det Aᵢ, las matrices Aᵢ, la solución (o None), el veredicto
    ('única', 'sin solución' o 'usar el rango') y un mensaje. Si det A = 0 no divide."""
    Ab = aumentada(Ab)
    m, n = Ab.rows, Ab.cols - 1
    if m != n:
        raise ValueError(f"la regla de Cramer solo se aplica a sistemas cuadrados (tantas ecuaciones como incógnitas); "
                         f"este tiene {_ecuaciones(m)} y {_incognitas(n)}: resuélvelo con Gauss-Jordan (sección 1) "
                         "o clasifícalo con los rangos (sección 4)")
    A, b = Ab[:, :n], Ab[:, n]
    d = A.det()
    matrices, dets = [], []
    for i in range(n):
        Ai = A.copy()
        Ai[:, i] = b
        matrices.append(Ai)
        dets.append(Ai.det())
    nombres = nombres_incognitas(n)
    if d != 0:
        x = sp.Matrix([di / d for di in dets])
        veredicto, mensaje = "única", f"det A = {d} ≠ 0: el sistema tiene una sola solución"
    else:
        x = None
        distintos = [i for i in range(n) if dets[i] != 0]
        if distintos:
            i = distintos[0]
            veredicto = "sin solución"
            mensaje = (f"det A = 0 y det A{_sub(i + 1)} = {dets[i]} ≠ 0: no hay solución. Si la hubiera, "
                       f"det A{_sub(i + 1)} = {nombres[i]}·det A = {nombres[i]}·0 = 0, y no es 0")
        else:
            veredicto = "usar el rango"
            mensaje = ("det A = 0 y todos los det Aᵢ valen 0: con Cramer no se puede decidir. El sistema puede tener "
                       "infinitas soluciones o ninguna; hay que usar el rango: compara rango(A) con rango[A | b] (sección 4)")
    res = {"det_A": d, "det_Ai": dets, "Ai": matrices, "solucion": x, "veredicto": veredicto, "mensaje": mensaje}
    if pasos:
        _informe_cramer(Ab, res, n_cifras)
    return res


def _informe_cramer(Ab, res, n_cifras):
    n = Ab.cols - 1
    A, b = Ab[:, :n], Ab[:, n]
    nombres = nombres_incognitas(n)
    _imprime_sistema(Ab)
    muestra("A", A)
    muestra("b", b)
    muestra("det A", res["det_A"], n_cifras)
    for i, (Ai, di) in enumerate(zip(res["Ai"], res["det_Ai"]), 1):
        print(f"A{_sub(i)}: la columna {i} de A cambiada por b")
        muestra(f"A{_sub(i)}", Ai)
        muestra(f"det A{_sub(i)}", di, n_cifras)
    print(res["mensaje"] + ".")
    if res["solucion"] is None:
        return
    d = res["det_A"]
    for i, (v, di, xi) in enumerate(zip(nombres, res["det_Ai"], res["solucion"]), 1):
        decimal = "" if xi.is_integer else f" ≈ {cifras(xi, n_cifras)}"
        print(f"  {v} = det A{_sub(i)} / det A = {_paren(di)}/{_paren(d)} = {xi}{decimal}")
    print("Comprobación: A x = b" if A * res["solucion"] == b else "Comprobación FALLÓ: A x ≠ b")


def costos(n):
    """(Cramer con cofactores, Gauss): multiplicaciones y divisiones para un sistema n×n, sin aprovechar ceros.
    Cramer: (n+1)·Mₙ + n con Mₙ = n·(Mₙ₋₁ + 1), M₂ = 2 (y M₁ = 0). Gauss: (n³ + 3n² - n)/3, que siempre es entero."""
    if isinstance(n, bool) or not isinstance(n, (int, np.integer, sp.Integer)) or int(n) < 1:
        raise ValueError(f"n = {n} no sirve: el tamaño del sistema debe ser un entero positivo")
    n = int(n)
    M = 0                                   # M₁ = 0: el determinante de una 1×1 es su única entrada
    for k in range(2, n + 1):
        M = k * (M + 1)                     # M₂ = 2, M₃ = 9, M₄ = 40, ...
    gauss, resto = divmod(n ** 3 + 3 * n ** 2 - n, 3)
    assert resto == 0
    return (n + 1) * M + n, gauss


def tabla_costos(n_sistema=None, tamanos=(2, 3, 4, 5, 6, 10)):
    """Imprime la tabla de operaciones de Cramer y de Gauss; marca el tamaño de tu sistema."""
    filas = sorted(set(tamanos) | ({n_sistema} if n_sistema else set()))
    print("Operaciones (multiplicaciones y divisiones, sin aprovechar ceros):")
    print(f"{'n':>4}  {'Cramer con cofactores':>22}  {'Gauss':>7}")
    for k in filas:
        c, g = costos(k)
        marca = "   ← tu sistema" if k == n_sistema else ""
        print(f"{k:>4}  {c:>22,}  {g:>7,}{marca}".replace(",", " "))

**Recorrido paso a paso.** El sistema $2x+y=5$, $x+3y=5$: primero $\det A$, luego cada $A_i$ (la columna $i$ cambiada por $\mathbf b$) y su determinante, y al final los cocientes. Después, la tabla de costos.

In [ ]:
_recorrido = cramer("2, 1, 5; 1, 3, 5")
print()
tabla_costos()

**Calculadora.** Escribe un sistema cuadrado como matriz aumentada (la última columna es $\mathbf b$). Prueba también uno con $\det A=0$, por ejemplo `1, 2, 3; 2, 4, 7`.

In [ ]:
def calculadora_cramer(Ab_txt, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            Ab = aumentada(Ab_txt)
            res = cramer(Ab, pasos=False)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_cramer(Ab, res, n_cifras)
        print()
        tabla_costos(Ab.rows)


widgets.interact(calculadora_cramer,
    Ab_txt=widgets.Textarea(value="3, 2, 7\n1, -1, -1", description="[A | b] =", continuous_update=False,
                            layout=widgets.Layout(width="340px", height="110px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
_CR = lambda texto: cramer(texto, pasos=False)
PRUEBAS_2 = [
    ("2, 1, 5; 1, 3, 5 → det A = 5, det A₁ = 10, det A₂ = 5, x = (2, 1)",
     lambda: (lambda r: r["det_A"] == 5 and r["det_Ai"] == [10, 5] and r["solucion"] == vector("2, 1"))(_CR("2, 1, 5; 1, 3, 5"))),
    ("3, 2, 7; 1, -1, -1 → det A = -5, x = (1, 2)",
     lambda: (lambda r: r["det_A"] == -5 and r["solucion"] == vector("1, 2"))(_CR("3, 2, 7; 1, -1, -1"))),
    ("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13 → det A = -2, det A₃ = -6, x = (1, 2, 3)",
     lambda: (lambda r: r["det_A"] == -2 and r["det_Ai"][2] == -6 and r["solucion"] == vector("1, 2, 3"))(
         _CR("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13"))),
    ("1, 2, 3; 2, 4, 7 → det A = 0 y det A₁ = -2: no hay solución (sin dividir entre 0)",
     lambda: (lambda r: r["det_A"] == 0 and r["det_Ai"][0] == -2 and r["solucion"] is None
                        and r["veredicto"] == "sin solución" and "no hay solución" in r["mensaje"])(_CR("1, 2, 3; 2, 4, 7"))),
    ("1, 2, 3; 2, 4, 6 → det A = 0 y todos los det Aᵢ = 0: hay que usar el rango",
     lambda: (lambda r: r["det_A"] == 0 and r["det_Ai"] == [0, 0] and r["veredicto"] == "usar el rango"
                        and "hay que usar el rango" in r["mensaje"])(_CR("1, 2, 3; 2, 4, 6"))),
    ("planos paralelos 1, 1, 1, 1; 1, 1, 1, 2; 1, 1, 1, 3 → todos los determinantes 0: hay que usar el rango",
     lambda: (lambda r: r["det_A"] == 0 and r["det_Ai"] == [0, 0, 0] and r["veredicto"] == "usar el rango"
                        and "hay que usar el rango" in r["mensaje"])(_CR("1, 1, 1, 1; 1, 1, 1, 2; 1, 1, 1, 3"))),
    ("costos(2) = (8, 6), costos(3) = (39, 17), costos(4) = (204, 36), costos(10) = (68588310, 430)",
     lambda: [costos(k) for k in (2, 3, 4, 10)] == [(8, 6), (39, 17), (204, 36), (68588310, 430)]),
    ("0.5, 0.3, 1.9; 0.2, 0.4, 1.4 → x = (17/7, 16/7) exacto",
     lambda: _CR("0.5, 0.3, 1.9; 0.2, 0.4, 1.4")["solucion"] == sp.Matrix([sp.Rational(17, 7), sp.Rational(16, 7)])),
]
corre_pruebas(PRUEBAS_2, 2)

**Contrasta (sección 2).** Resuelve con Cramer $2x+5y=1$, $x+3y=2$: calcula $\det A$, $\det A_1$ y $\det A_2$ y divide. Escribe tu solución (primero $x$, luego $y$) y, si quieres, tu $\det A$.

In [ ]:
mi_solucion = None     # escribe tu solución como texto, primero x y luego y; por ejemplo: "4, -1"
mi_det_A = None        # opcional: tu det A, por ejemplo "-3"

if mi_solucion is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mia = vector(str(mi_solucion), "tu solución")
        mio_det = None if mi_det_A is None else numero(mi_det_A, "tu det A")
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        if mia.rows != 2:          # respuesta incompleta: la referencia todavía no se muestra
            print(f"Tu solución debe tener dos números, primero x y luego y (escribiste {mia.rows}): "
                  "complétala y vuelve a ejecutar la celda.")
        else:
            ref = cramer("2, 5, 1; 1, 3, 2", pasos=False)
            print(f"La calculadora da: det A = {ref['det_A']}; x = {ref['solucion'][0]}, y = {ref['solucion'][1]}")
            if mio_det is not None:
                print("det A:", "coincide" if mio_det == ref["det_A"] else
                      "NO coincide: det A = a₁₁a₂₂ - a₁₂a₂₁ (el producto de la diagonal menos el de la otra diagonal)")
            print("Solución:", compara_vector(mia, ref["solucion"], ["x", "y"],
                                              "Para det A₁ cambia la columna de x por b = (1, 2); para det A₂, la de y."))
            print("\nEl procedimiento completo:")
            cramer("2, 5, 1; 1, 3, 2")

## 3. Ecuaciones vectorial y matricial; rango

El mismo sistema se escribe de tres maneras. Con $\mathbf a_j$ la columna $j$ de $A$:

$$\text{ecuaciones: }\ a_{i1}x_1+\dots+a_{in}x_n=b_i\ \ (i=1,\dots,m),\qquad \text{matricial: }\ A\mathbf x=\mathbf b,\qquad \text{vectorial: }\ x_1\mathbf a_1+x_2\mathbf a_2+\dots+x_n\mathbf a_n=\mathbf b.$$

- **Lectura por renglones.** Cada ecuación es una recta (o un plano) y la solución es el punto común a todas.
- **Lectura por columnas.** El sistema pregunta con qué coeficientes hay que combinar las columnas de $A$ para llegar a $\mathbf b$. Una suma $c_1\mathbf a_1+\dots+c_n\mathbf a_n$ es una **combinación lineal** de las columnas.

Por ejemplo, $\begin{bmatrix}2&1\\1&3\end{bmatrix}\begin{bmatrix}2\\1\end{bmatrix}=2\begin{bmatrix}2\\1\end{bmatrix}+1\begin{bmatrix}1\\3\end{bmatrix}=\begin{bmatrix}5\\5\end{bmatrix}$: el producto $A\mathbf x$ **es** la combinación de las columnas con los coeficientes de $\mathbf x$. Por eso:

> $A\mathbf x=\mathbf b$ tiene solución si y solo si $\mathbf b$ es combinación lineal de las columnas de $A$, y los coeficientes de la combinación son la solución.

**Rango.** $\operatorname{rango}(A)$ es el número de pivotes de una forma escalonada de $A$, es decir, el número de renglones que no quedan en ceros al escalonar. Las columnas donde caen los pivotes son las **columnas pivote**.

- No depende de qué operaciones elijas para escalonar.
- Si $A$ es $m\times n$, $\operatorname{rango}(A)\le\min(m,n)$: cada pivote ocupa un renglón y una columna distintos.
- Es el número máximo de renglones de $A$ entre los que ninguno es combinación de los otros, y coincide con el de columnas.
- Si $A$ es $n\times n$: $\operatorname{rango}(A)=n$ si y solo si $\det A\neq 0$, si y solo si $A$ es invertible.

**Dos rangos.** Con $r=\operatorname{rango}(A)$ y $r^*=\operatorname{rango}[A\,|\,\mathbf b]$, siempre $r\le r^*\le r+1$. Si $r^*=r$, $\mathbf b$ es combinación de las columnas y el sistema tiene solución. Si $r^*=r+1$, no lo es: al escalonar $[A\,|\,\mathbf b]$ aparece un renglón $0=c$ con $c\neq0$.

`numpy.linalg.matrix_rank` calcula el rango en punto flotante, con una tolerancia; la calculadora lo compara con el rango exacto.

In [ ]:
def _solo_gauss(pasos):
    """Los pasos de la fase de Gauss (hasta la forma escalonada), sin los de Jordan."""
    fases = [k for k, (tipo, _, _) in enumerate(pasos) if tipo == "fase"]
    return pasos[:fases[1]] if len(fases) > 1 else pasos


def rango_y_pivotes(A, pasos=False):
    """(forma escalonada exacta, columnas pivote numeradas desde 1, rango) de una matriz A de hasta 6×6."""
    A = matriz_limitada(A, "A")
    el = _elimina(A, A.cols, con_b=False)
    if pasos:
        muestra("A", A)
        _imprime_pasos(_solo_gauss(el["pasos"]))
    return el["E"], [c + 1 for c in el["pivotes"]], el["r"]


def combinacion(A, b):
    """¿Es b combinación lineal de las columnas de A? Diccionario con 'es' (True o False), los rangos r y r_aum,
    las formas escalonada y reducida de [A | b], los coeficientes (con parámetros s, t, ... si hay infinitas
    combinaciones), la particular y las direcciones, y, si no lo es, el renglón '0 = c' que lo prueba."""
    A, b = matriz_limitada(A, "A"), vector(b, "b")
    if b.rows != A.rows:
        raise ValueError(f"b tiene {_entradas(b.rows)} y A tiene {A.rows} renglones: deben coincidir")
    n = A.cols
    el = _elimina(A.row_join(b), n, con_b=True)
    res = {"es": el["imposible"] is None, "r": el["r"], "r_aum": el["r_aum"], "escalonada": el["E"],
           "reducida": el["R"], "pivotes": [c + 1 for c in el["pivotes"]], "coeficientes": None, "particular": None,
           "direcciones": [], "parametros": [], "renglon": None}
    if el["imposible"] is not None:
        f, c = el["imposible"]
        res["renglon"] = f"R{f} de la forma escalonada de [A | b] dice 0 = {c}"
    else:
        x, xp, dirs, ps = solucion_general(el["R"], el["pivotes"], n)
        res.update(coeficientes=x, particular=xp, direcciones=dirs, parametros=ps)
    return res


def _texto_combinacion(coef, ps=()):
    """'2·a₁ + 1·a₂' o, con parámetros, '(3 - 2s)·a₁ + s·a₂' (la constante primero)."""
    partes = []
    for j, c in enumerate(coef, 1):
        c = sp.sympify(c)
        texto = _paren(c) if c.is_number else (str(c) if c.is_Symbol else f"({_lineal(c, ps)})")
        partes.append(f"{texto}·a{_sub(j)}")
    return " + ".join(partes)


def _flecha(ax, origen, punta, color, estilo, ancho, texto, desplazamiento=(6, 6)):
    if np.hypot(*(np.asarray(punta, float) - np.asarray(origen, float))) > 1e-12:
        ax.annotate("", xy=punta, xytext=origen,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=ancho, linestyle=estilo, shrinkA=0, shrinkB=0))
    ax.annotate(texto, xy=punta, xytext=desplazamiento, textcoords="offset points", fontsize=11)


def dibuja_columnas(A, b=None):
    """Lectura por columnas en el plano (A de 2×2): a₁, a₂, b y, si se puede, el camino x₁a₁ + x₂a₂ hasta b."""
    A = matriz_limitada(A, "A")
    if A.shape != (2, 2):
        raise ValueError(f"el dibujo de las columnas es para A de 2×2 (esta es {dims(A)})")
    a1, a2 = [_flotante(A[:, j]).ravel() for j in range(2)]
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    puntos = [np.zeros(2), a1, a2]
    _flecha(ax, (0, 0), a1, _COLORES[0], "-", 2, "a₁", (8, -14))
    _flecha(ax, (0, 0), a2, _COLORES[1], "--", 2, "a₂")
    titulo = "Columnas de A"
    if b is not None:
        res = combinacion(A, b)
        bf = _flotante(vector(b, "b")).ravel()
        puntos.append(bf)
        if res["es"]:
            c = res["particular"]                   # con infinitas combinaciones, la que tiene los parámetros en 0
            p1 = float(c[0]) * a1
            puntos.append(p1)
            ax.plot(*np.array([[0, 0], p1]).T, ":", color="0.35", lw=1.5)
            ax.plot(*np.array([p1, bf]).T, ":", color="0.35", lw=1.5, label="camino x₁a₁, luego + x₂a₂")
            ax.plot(*p1, "s", color="0.35", ms=6)
            titulo = f"b = {_texto_combinacion(c)}" + (" (una de infinitas)" if res["parametros"] else "")
        else:
            titulo = "b no es combinación de a₁ y a₂"
        if res["r"] == 1:                           # columnas en una misma recta: se dibuja esa recta
            d = a1 if np.hypot(*a1) > 0 else a2
            t = np.linspace(-1, 1, 2) * 2 * max(np.abs(np.vstack(puntos)).max(), 1) / np.hypot(*d)
            ax.plot(t * d[0], t * d[1], "-.", color="0.6", lw=1, label="recta de las combinaciones")
        _flecha(ax, (0, 0), bf, "black", "-", 3, "b")
    lim = 1.2 * max(1.0, float(np.abs(np.vstack(puntos)).max()))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.grid(alpha=0.3); ax.set_xlabel("primera componente"); ax.set_ylabel("segunda componente")
    ax.set_title(titulo)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower right", fontsize=8)
    plt.show()

**Recorrido paso a paso.** Primero, $A\mathbf x$ como combinación de columnas. Después, el rango de $\begin{bmatrix}1&2&3\\2&4&7\\3&6&10\end{bmatrix}$: al escalonar, la columna 2 se queda sin pivote porque es el doble de la columna 1.

In [ ]:
_A = matriz("2, 1; 1, 3")
_x = vector("2, 1")
muestra("A x", _A * _x)
muestra("2·a₁ + 1·a₂", 2 * _A[:, 0] + 1 * _A[:, 1])
print("Son el mismo vector: A x es la combinación de las columnas con los coeficientes de x.\n")
_E, _piv, _r = rango_y_pivotes("1, 2, 3; 2, 4, 7; 3, 6, 10", pasos=True)
muestra("forma escalonada", _E)
print(f"Columnas pivote: {_lista(_piv)} → rango = {_r}; numpy.linalg.matrix_rank da",
      np.linalg.matrix_rank(_flotante(matriz("1, 2, 3; 2, 4, 7; 3, 6, 10"))))
dibuja_columnas(_A, _A * _x)

**Calculadora.** Escribe $A$ por renglones y, si quieres, $\mathbf b$ (déjalo vacío para calcular solo el rango de $A$). Si $A$ es $2\times 2$, puedes dibujar la lectura por columnas.

In [ ]:
def _informe_rango(A_txt, b_txt, n_cifras):
    """Imprime el análisis y devuelve (A, b) para el dibujo, o None si la entrada tiene un error."""
    try:
        A = matriz_limitada(A_txt, "A")
        b = vector(b_txt, "b") if b_txt.strip() else None
        res = combinacion(A, b) if b is not None else None
    except ValueError as err:
        print("Revisa la entrada:", err); return None
    muestra("A", A, n_cifras)
    E, piv, r = rango_y_pivotes(A)
    muestra("forma escalonada de A", E)
    r_np = int(np.linalg.matrix_rank(_flotante(A)))
    print(f"Columnas pivote: {_lista(piv) if piv else 'ninguna'} → rango(A) = {r}")
    print(f"numpy.linalg.matrix_rank(A) = {r_np}" + (" (coincide)" if r_np == r else
          " (NO coincide: numpy usa una tolerancia en punto flotante; el rango exacto es el de la forma escalonada)"))
    if b is None:
        return A, None
    Ab = A.row_join(b)
    muestra_aumentada("forma escalonada de [A | b]", res["escalonada"], A.cols)
    r_np2 = int(np.linalg.matrix_rank(_flotante(Ab)))
    print(f"rango[A | b] = {res['r_aum']}; numpy.linalg.matrix_rank([A | b]) = {r_np2}" +
          (" (coincide)" if r_np2 == res["r_aum"] else " (NO coincide: numpy usa una tolerancia en punto flotante)"))
    if res["es"]:
        print(f"rango[A | b] = rango(A) = {r}: b SÍ es combinación de las columnas de A.")
        if res["parametros"]:
            print(f"Hay infinitas combinaciones ({len(res['parametros'])} parámetro{'s' if len(res['parametros']) > 1 else ''}, "
                  f"{_coma(res['parametros'])}): b = {_texto_combinacion(res['coeficientes'], res['parametros'])}")
            print(f"Por ejemplo, con {'todos los parámetros en 0' if len(res['parametros']) > 1 else str(res['parametros'][0]) + ' = 0'}: "
                  f"b = {_texto_combinacion(res['particular'])}")
        else:
            print(f"Una sola combinación: b = {_texto_combinacion(res['coeficientes'])}")
        print("Comprobación: A·(coeficientes) = b" if A * res["particular"] == b else "Comprobación FALLÓ")
    else:
        print(f"rango[A | b] = {res['r_aum']} = rango(A) + 1: b NO es combinación de las columnas de A; "
              f"{res['renglon']}.")
    return A, b


def calculadora_rango(A_txt, b_txt, dibujar, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        dibujo = _informe_rango(A_txt, b_txt, n_cifras)
    if dibujar and dibujo is not None and dibujo[0].shape == (2, 2):
        dibuja_columnas(*dibujo)


widgets.interact(calculadora_rango,
    A_txt=widgets.Textarea(value="1, 2\n2, 4", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="340px", height="110px")),
    b_txt=widgets.Text(value="3, 6", description="b =", continuous_update=False),
    dibujar=widgets.Checkbox(value=True, description="dibujar columnas (A de 2×2)"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
_A13 = de_columnas("1, 0, 1", "0, 1, 1")
PRUEBAS_3 = [
    ("1, 2, 3; 2, 4, 7; 3, 6, 10 → rango 2, columnas pivote [1, 3]",
     lambda: rango_y_pivotes("1, 2, 3; 2, 4, 7; 3, 6, 10")[1:] == ([1, 3], 2)),
    ("identidad 3×3 → rango 3", lambda: rango_y_pivotes(sp.eye(3))[2] == 3),
    ("matriz de ceros 2×3 → rango 0 (ninguna columna pivote)", lambda: rango_y_pivotes(sp.zeros(2, 3))[1:] == ([], 0)),
    ("1, 2; 2, 4; 3, 6 → rango 1", lambda: rango_y_pivotes("1, 2; 2, 4; 3, 6")[2] == 1),
    ("1, 0, 2, 1; 0, 1, 1, 3 → rango 2", lambda: rango_y_pivotes("1, 0, 2, 1; 0, 1, 1, 3")[2] == 2),
    ("a₁ = (1, 0, 1), a₂ = (0, 1, 1), b = (1, 1, 2) → b = 1·a₁ + 1·a₂",
     lambda: (lambda r: r["es"] and r["coeficientes"] == vector("1, 1") and not r["parametros"])(combinacion(_A13, "1, 1, 2"))),
    ("mismas columnas, b = (1, 1, 3) → no es combinación: rango A = 2, rango [A|b] = 3, renglón 0 = 1",
     lambda: (lambda r: not r["es"] and (r["r"], r["r_aum"]) == (2, 3) and "0 = 1" in r["renglon"])(combinacion(_A13, "1, 1, 3"))),
    ("0.1, 0.2; 0.3, 0.6 → rango 1 con fracciones exactas, igual que numpy.linalg.matrix_rank",
     lambda: (lambda E, piv, r: r == 1 and all(e.is_Rational for e in E)
                                and np.linalg.matrix_rank(_flotante(matriz("0.1, 0.2; 0.3, 0.6"))) == 1)(
         *rango_y_pivotes("0.1, 0.2; 0.3, 0.6"))),
]
corre_pruebas(PRUEBAS_3, 3)

**Contrasta (sección 3).** Calcula a mano el rango de $\begin{bmatrix}1&2&3\\0&1&1\\1&3&4\end{bmatrix}$ escalonando, y escribe el número.

In [ ]:
mi_rango = None     # escribe un número entero, por ejemplo 3

if mi_rango is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = numero(mi_rango, "tu rango")
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        if not (mio.is_integer and 0 <= mio <= 3):     # respuesta imposible: la referencia todavía no se muestra
            print(f"El rango de una matriz 3×3 es un entero de 0 a 3 (escribiste {mio}): revisa tu número "
                  "y vuelve a ejecutar la celda.")
        else:
            E, piv, ref = rango_y_pivotes("1, 2, 3; 0, 1, 1; 1, 3, 4")
            print("La calculadora da: rango =", ref)
            print("coinciden" if mio == ref else
                  "NO coinciden: cuenta los renglones que no quedan en ceros después de escalonar; "
                  "fíjate si un renglón es combinación de los otros.")
            print("\nEl procedimiento completo:")
            rango_y_pivotes("1, 2, 3; 0, 1, 1; 1, 3, 4", pasos=True)
            muestra("forma escalonada", E)

## 4. Clasificación y sistemas homogéneos

Con dos rangos, el de $A$ y el de la aumentada, clasificas cualquier sistema sin terminar de resolverlo.

**Teorema de Rouché-Frobenius.** Sea $A$ de $m\times n$, $r=\operatorname{rango}(A)$ y $r^*=\operatorname{rango}[A\,|\,\mathbf b]$. Siempre $r\le r^*\le r+1$, y:

| Rangos | Clasificación | Soluciones |
|---|---|---|
| $r<r^*$ | incompatible | ninguna: al escalonar aparece un renglón $0=c$, con $c\neq0$ |
| $r=r^*=n$ | compatible determinado | una sola |
| $r=r^*<n$ | compatible indeterminado | infinitas, que dependen de $n-r$ parámetros |

**Variables básicas y libres.** En la forma escalonada de un sistema compatible, las incógnitas de las columnas con pivote son *básicas* y las demás son *libres*. La **solución general** da cada básica en función de las libres, que se renombran como parámetros.

*Convención de este notebook:* la primera variable libre (de izquierda a derecha) se llama $s$, la segunda $t$, y siguen $u$, $v$, $w$ (con más de cinco, $s_1, s_2,\dots$). Cada parámetro **es** el valor de su variable libre: si $y$ es libre, $y=s$.

**Particular más homogénea.** Si $\mathbf x_p$ resuelve $A\mathbf x=\mathbf b$, todas las soluciones son $\mathbf x=\mathbf x_p+\mathbf x_h$, con $A\mathbf x_h=\mathbf 0$, porque la diferencia de dos soluciones cumple $A(\mathbf x-\mathbf x_p)=\mathbf b-\mathbf b=\mathbf 0$. Por eso un sistema compatible tiene una solución o infinitas, nunca dos ni tres. La calculadora escribe la solución general así: $\mathbf x=\mathbf x_p+s\,\mathbf d_1+t\,\mathbf d_2+\cdots$, con $\mathbf x_p$ la solución que tiene los parámetros en 0.

**Sistemas homogéneos** ($\mathbf b=\mathbf 0$, es decir, $A\mathbf x=\mathbf 0$). Siempre son compatibles, porque $\mathbf x=\mathbf 0$ (la solución *trivial*) los cumple. Tienen soluciones distintas de la trivial si y solo si $r<n$; en particular, siempre que haya menos ecuaciones que incógnitas y, si $A$ es cuadrada, cuando $\det A=0$. Todas las soluciones se obtienen dando valores a los parámetros de la solución general: con todos en cero sale la trivial, y con cualquier otro valor, una no trivial. Como la particular es $\mathbf 0$, la solución general es $s\,\mathbf d_1+t\,\mathbf d_2+\cdots$; la calculadora muestra además cada dirección $\mathbf d_i$ escalada para que su primera componente distinta de cero sea 1.

**Ojo.** No clasifiques contando ecuaciones: cuatro ecuaciones con tres incógnitas pueden tener solución única, y tres con tres pueden no tener ninguna. Tampoco uses `numpy.linalg.lstsq`: devuelve un vector aunque el sistema sea incompatible, sin avisar. La calculadora decide con los rangos exactos.

In [ ]:
def _escala_primera(d):
    """El vector d dividido entre su primera componente distinta de cero (para que esa componente sea 1)."""
    primera = next(e for e in d if e != 0)
    return d / primera


def clasificar(Ab, explicar=False, pasos=False, n_cifras=4):
    """Clasifica [A | b] con el teorema de Rouché-Frobenius, con rangos exactos (sin lstsq ni pinv).

    Devuelve un diccionario con m, n, r, r_aum, tipo ('incompatible', 'compatible determinado' o 'compatible
    indeterminado'), num_parametros, las formas escalonada y reducida, la solución (única, o general con los
    parámetros s, t, u, ... en el orden de las variables libres), la particular, las direcciones, el renglón (R, c)
    de 0 = c si es incompatible (tomado de la forma escalonada) y, si es homogéneo, las direcciones escaladas para que
    su primera componente distinta de cero sea 1. Con explicar=True imprime el informe; con pasos=True, también
    cada operación."""
    Ab = aumentada(Ab)
    m, n = Ab.rows, Ab.cols - 1
    A, b = Ab[:, :n], Ab[:, n]
    el = _elimina(Ab, n, con_b=True)
    r, r_aum = el["r"], el["r_aum"]
    if r < r_aum:
        tipo = "incompatible"
    elif r == n:
        tipo = "compatible determinado"
    else:
        tipo = "compatible indeterminado"
    res = {"m": m, "n": n, "r": r, "r_aum": r_aum, "tipo": tipo, "num_parametros": 0 if tipo == "incompatible" else n - r,
           "escalonada": el["E"], "reducida": el["R"], "pivotes": [c + 1 for c in el["pivotes"]],
           "libres": [c + 1 for c in el["libres"]], "homogeneo": b == sp.zeros(m, 1), "solucion": None,
           "particular": None, "direcciones": [], "parametros": [], "renglon": el["imposible"], "escaladas": [],
           "residuo": None, "pasos": el["pasos"]}
    if tipo != "incompatible":
        x, xp, dirs, ps = solucion_general(el["R"], el["pivotes"], n)
        res.update(solucion=x, particular=xp, direcciones=dirs, parametros=ps, residuo=b - A * xp)
        if res["homogeneo"]:
            res["escaladas"] = [_escala_primera(d) for d in dirs]
    if explicar:
        _informe_clasificacion(Ab, res, n_cifras, pasos)
    return res


def _informe_clasificacion(Ab, res, n_cifras, pasos=False):
    m, n, r, r_aum = res["m"], res["n"], res["r"], res["r_aum"]
    A = Ab[:, :n]
    nombres = nombres_incognitas(n)
    _imprime_sistema(Ab)
    muestra_aumentada("[A | b]", Ab, n)
    if pasos:
        _imprime_pasos(res["pasos"], corte=n)
    muestra_aumentada("forma escalonada", res["escalonada"], n)
    muestra_aumentada("forma escalonada reducida", res["reducida"], n)
    print(f"r = rango(A) = {r},  r* = rango[A | b] = {r_aum},  n = {n} (incógnitas)")
    if res["tipo"] == "incompatible":
        f, c = res["renglon"]
        print(f"r < r*: el sistema es INCOMPATIBLE (teorema de Rouché-Frobenius). El renglón R{f} de la forma escalonada "
              f"dice 0 = {c}, y ningún valor de las incógnitas lo cumple.")
        if c != 1:
            print("En la forma escalonada reducida ese renglón queda como 0 = 1.")
        return
    if res["homogeneo"]:
        print("Sistema homogéneo (b = 0): siempre es compatible, porque x = 0 (la solución trivial) lo cumple.")
    if res["tipo"] == "compatible determinado":
        x = res["solucion"]
        decimales = "" if all(e.is_integer for e in x) else " ≈ " + _aprox(x, n_cifras)
        print(f"r = r* = n = {n}: COMPATIBLE DETERMINADO, una sola solución.")
        print("Solución: " + ", ".join(f"{v} = {e}" for v, e in zip(nombres, x)) + decimales)
        if res["homogeneo"]:
            print(f"Solo tiene la solución trivial (r = n = {n}): no hay soluciones no triviales.")
        print("Comprobación: b - A x =", _tupla(res["residuo"]))
        return
    k = res["num_parametros"]
    ps = res["parametros"]
    libres = [nombres[c - 1] for c in res["libres"]]
    basicas = [nombres[c - 1] for c in res["pivotes"]]
    print(f"r = r* = {r} < n = {n}: COMPATIBLE INDETERMINADO, infinitas soluciones con n - r = {k} "
          f"parámetro{'s' if k > 1 else ''}.")
    print(f"Variables básicas (columnas con pivote): {_coma(basicas) if basicas else 'ninguna'}. "
          f"Libres: {_coma(libres)} → " + ", ".join(f"{v} = {p}" for v, p in zip(libres, ps)) + ".")
    print("Solución general:")
    for v, e in zip(nombres, res["solucion"]):
        print(f"  {v} = {_lineal(e, ps)}")
    vec = "(" + ", ".join(nombres) + ")"
    suma = " + ".join(f"{p}·{_tupla(d)}" for p, d in zip(ps, res["direcciones"]))
    cero = res["particular"] == sp.zeros(n, 1)
    print(f"Forma vectorial: {vec} = " + (suma if cero else f"{_tupla(res['particular'])} + {suma}"))
    ceros = all(A * d == sp.zeros(m, 1) for d in res["direcciones"])
    print(f"Comprobación: con {'todos los parámetros en 0' if k > 1 else f'{ps[0]} = 0'}, la particular {_tupla(res['particular'])} da "
          f"b - A x = {_tupla(res['residuo'])}; " +
          ("cada dirección d cumple A d = 0 (es solución del homogéneo)." if ceros else "una dirección NO cumple A d = 0."))
    if res["homogeneo"]:
        print(f"Hay soluciones no triviales (r = {r} < n = {n}): todas las soluciones se obtienen dando valores a "
              f"{'los parámetros' if k > 1 else ps[0]}; con {'todos en cero' if k > 1 else str(ps[0]) + ' = 0'} sale la trivial.")
        escaladas = ", ".join(_tupla(v) for v in res["escaladas"])
        print(f"Con {'cada dirección escalada' if k > 1 else 'la dirección escalada'} para que su primera componente "
              f"distinta de cero sea 1: {escaladas}. Las soluciones son " +
              (f"los múltiplos de {escaladas}." if k == 1 else "las sumas de múltiplos de esos vectores."))


_CLASIFICACIONES = {      # respuestas que se aceptan, ya normalizadas (minúsculas, sin acentos ni puntuación)
    "incompatible": ["incompatible", "es incompatible", "sistema incompatible", "sin solucion", "no tiene solucion",
                     "no hay solucion", "ninguna solucion", "ninguna"],
    "compatible determinado": ["determinado", "determinada", "compatible determinado", "compatible determinada",
                               "unica", "solucion unica", "una solucion", "una sola solucion", "tiene solucion unica"],
    "compatible indeterminado": ["indeterminado", "indeterminada", "compatible indeterminado", "compatible indeterminada",
                                 "infinitas", "infinitas soluciones", "tiene infinitas soluciones"],
}


def _tipo_de_texto(texto):
    """La clasificación que escribió el alumno, comparando la frase completa (así 'no es única' no se confunde con
    'única'). ValueError si no es una de las respuestas que se aceptan."""
    t = normaliza_texto(texto)
    for tipo, frases in _CLASIFICACIONES.items():
        if t in frases:
            return tipo
    raise ValueError(f"«{texto}» no es una de las respuestas que reconoce esta celda: escribe incompatible "
                     "(o sin solución), determinado (o solución única) o indeterminado (o infinitas soluciones)")

**Recorrido paso a paso.** Tres sistemas, uno de cada tipo: $x+y+z=6$, $2x+2y+3z=15$, $x+3y+2z=13$; luego $x+2y-z=3$, $2x+4y+z=9$; luego $x+y+z=1$, $2x+2y+2z=3$, $x-y=0$. Después, el sistema homogéneo del balance de la combustión del propano, $a\,\mathrm{C_3H_8}+b\,\mathrm{O_2}\to c\,\mathrm{CO_2}+d\,\mathrm{H_2O}$: carbono $3a-c=0$, hidrógeno $8a-2d=0$, oxígeno $2b-2c-d=0$ (aquí $a,b,c,d$ son $x_1,\dots,x_4$).

In [ ]:
for _texto in ("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13", "1, 2, -1, 3; 2, 4, 1, 9", "1, 1, 1, 1; 2, 2, 2, 3; 1, -1, 0, 0"):
    clasificar(_texto, explicar=True)
    print("\n" + "=" * 90 + "\n")
clasificar("3, 0, -1, 0, 0; 8, 0, 0, -2, 0; 0, 2, -2, -1, 0", explicar=True);

**Calculadora.** Escribe la matriz aumentada (la última columna es $\mathbf b$; para un sistema homogéneo, ceros). Prueba sistemas con más ecuaciones que incógnitas y al revés.

In [ ]:
def calculadora_clasificacion(Ab_txt, ver_pasos, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            Ab = aumentada(Ab_txt)
            res = clasificar(Ab)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_clasificacion(Ab, res, n_cifras, pasos=ver_pasos)


widgets.interact(calculadora_clasificacion,
    Ab_txt=widgets.Textarea(value="1, 1, 1, 1, 4\n1, -1, 1, -1, 0", description="[A | b] =", continuous_update=False,
                            layout=widgets.Layout(width="340px", height="120px")),
    ver_pasos=widgets.Checkbox(value=False, description="mostrar cada operación"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
_s = sp.Symbol("s")
_CL = lambda texto: clasificar(texto)
_cumple = lambda texto, r: sp.expand(aumentada(texto)[:, :-1] * r["solucion"] - aumentada(texto)[:, -1]) == sp.zeros(r["m"], 1)
PRUEBAS_4 = [
    ("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13 → compatible determinado, (1, 2, 3)",
     lambda: (lambda r: r["tipo"] == "compatible determinado" and r["solucion"] == vector("1, 2, 3"))(
         _CL("1, 1, 1, 6; 2, 2, 3, 15; 1, 3, 2, 13"))),
    ("1, 2, -1, 3; 2, 4, 1, 9 → indeterminado, 1 parámetro: x = 4 - 2s, y = s, z = 1",
     lambda: (lambda r: r["tipo"] == "compatible indeterminado" and r["num_parametros"] == 1
                        and r["solucion"] == sp.Matrix([4 - 2 * _s, _s, 1]))(_CL("1, 2, -1, 3; 2, 4, 1, 9"))),
    ("1, 1, 1, 1; 2, 2, 2, 3; 1, -1, 0, 0 → incompatible, r = 2, r* = 3",
     lambda: (lambda r: r["tipo"] == "incompatible" and (r["r"], r["r_aum"]) == (2, 3) and r["solucion"] is None)(
         _CL("1, 1, 1, 1; 2, 2, 2, 3; 1, -1, 0, 0"))),
    ("homogéneo 3, 0, -1, 0, 0; 8, 0, 0, -2, 0; 0, 2, -2, -1, 0 → soluciones múltiplos de (1, 5, 3, 4)",
     lambda: (lambda r: r["homogeneo"] and r["tipo"] == "compatible indeterminado"
                        and r["escaladas"] == [vector("1, 5, 3, 4")])(_CL("3, 0, -1, 0, 0; 8, 0, 0, -2, 0; 0, 2, -2, -1, 0"))),
    ("homogéneo 1, 2, 0, 0; 0, 1, 0, 0; 0, 0, 1, 0 → solo la solución trivial",
     lambda: (lambda r: r["homogeneo"] and r["tipo"] == "compatible determinado" and r["escaladas"] == []
                        and r["solucion"] == sp.zeros(3, 1))(_CL("1, 2, 0, 0; 0, 1, 0, 0; 0, 0, 1, 0"))),
    ("1, 1, 1, 1, 4; 1, -1, 1, -1, 0 → indeterminado, 2 parámetros (s, t), y la solución general cumple el sistema",
     lambda: (lambda r: r["tipo"] == "compatible indeterminado" and r["num_parametros"] == 2
                        and [str(p) for p in r["parametros"]] == ["s", "t"]
                        and _cumple("1, 1, 1, 1, 4; 1, -1, 1, -1, 0", r))(_CL("1, 1, 1, 1, 4; 1, -1, 1, -1, 0"))),
    ("1, 0, 0, 1; 0, 1, 0, 2; 0, 0, 1, 3; 1, 1, 1, 6 (4 ecuaciones, 3 incógnitas) → determinado, (1, 2, 3)",
     lambda: (lambda r: r["tipo"] == "compatible determinado" and r["solucion"] == vector("1, 2, 3"))(
         _CL("1, 0, 0, 1; 0, 1, 0, 2; 0, 0, 1, 3; 1, 1, 1, 6"))),
    ("el mismo con 7 en lugar de 6 en el último renglón → incompatible, r = 3, r* = 4",
     lambda: (lambda r: r["tipo"] == "incompatible" and (r["r"], r["r_aum"]) == (3, 4))(
         _CL("1, 0, 0, 1; 0, 1, 0, 2; 0, 0, 1, 3; 1, 1, 1, 7"))),
    ("0, 0, 0; 0, 0, 0 (2 ecuaciones, 2 incógnitas, todo cero) → indeterminado con 2 parámetros, r = 0",
     lambda: (lambda r: r["tipo"] == "compatible indeterminado" and r["r"] == 0 and r["num_parametros"] == 2)(
         _CL("0, 0, 0; 0, 0, 0"))),
    ("0, 0, 5 → incompatible: el renglón dice 0 = 5",
     lambda: (lambda r: r["tipo"] == "incompatible" and r["renglon"] == (1, 5))(_CL("0, 0, 5"))),
]
corre_pruebas(PRUEBAS_4, 4)

**Contrasta (sección 4).** Clasifica a mano $x-y=2$, $-2x+2y=-4$ con los rangos. Escribe si es incompatible, determinado o indeterminado y, si es indeterminado, cuántos parámetros tiene la solución general.

In [ ]:
mi_clasificacion = None     # escribe "incompatible", "determinado" o "indeterminado" (también "sin solución", "solución única" o "infinitas soluciones")
mis_parametros = None       # si escribiste "indeterminado": cuántos parámetros, por ejemplo 2

if mi_clasificacion is None:     # la referencia se muestra después de tu cálculo
    print("falta tu clasificación: escríbela arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = _tipo_de_texto(mi_clasificacion)
        mis_p = None if mis_parametros is None else numero(mis_parametros, "tu número de parámetros")
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        if mio == "compatible indeterminado" and mis_p is None:     # respuesta incompleta: todavía sin referencia
            print("Si es indeterminado, escribe también cuántos parámetros tiene la solución general (mis_parametros) "
                  "y vuelve a ejecutar la celda.")
        elif mio == "compatible indeterminado" and not (mis_p.is_integer and mis_p >= 1):
            print(f"El número de parámetros es un entero de 1 en adelante (escribiste {mis_p}): revísalo y vuelve a "
                  "ejecutar la celda.")
        else:
            ref = clasificar("1, -1, 2; -2, 2, -4")
            texto_ref = ref["tipo"] + (f" con {ref['num_parametros']} parámetro{'s' if ref['num_parametros'] > 1 else ''}"
                                       if ref["num_parametros"] else "")
            print(f"La calculadora da: {texto_ref} (r = {ref['r']}, r* = {ref['r_aum']}, n = {ref['n']}).")
            if mio != ref["tipo"]:
                print("NO coinciden: escalona la aumentada y compara r = rango(A) con r* = rango[A | b] y con n.")
            elif ref["num_parametros"] and mis_p != ref["num_parametros"]:
                print("La clasificación coincide, pero el número de parámetros no: es n - r.")
            else:
                print("coinciden")
            print("\nEl informe completo:")
            clasificar("1, -1, 2; -2, 2, -4", explicar=True, pasos=True)

## 5. Interpretación geométrica en 2D y 3D

Con dos incógnitas, cada ecuación $ax+by=c$ es una **recta** del plano; con tres, cada ecuación $ax+by+cz=d$ es un **plano** del espacio, con normal $\mathbf n=(a,b,c)$. La clasificación de la sección 4 tiene entonces una imagen, y los rangos dicen cuál.

**Dos rectas** (sistema $2\times 2$):

| $r$ | $r^*$ | Configuración |
|:-:|:-:|---|
| 2 | 2 | las rectas se cortan en un punto ($\det A\neq 0$) |
| 1 | 1 | son la misma recta |
| 1 | 2 | son paralelas y distintas |

**Tres planos** (sistema $3\times 3$):

| $r$ | $r^*$ | Configuración |
|:-:|:-:|---|
| 3 | 3 | un punto común |
| 2 | 2 | una recta común: los tres planos pasan por una recta, o dos son el mismo y el tercero lo corta |
| 1 | 1 | los tres son el mismo plano |
| 2 | 3 | sin punto común: un **prisma** si ningún par de normales es paralelo; **dos planos paralelos cortados por un tercero** si exactamente un par lo es |
| 1 | 2 | sin punto común: **planos paralelos** (al menos dos distintos) |

Dos normales son paralelas si una es múltiplo de la otra, es decir, si $\mathbf n_i\times\mathbf n_j=\mathbf 0$. Con $r=1$ todas las normales son paralelas; con $r=2$ y $r^*=3$, la comparación de las normales separa el prisma de los planos paralelos cortados por un tercero.

Con otra cantidad de ecuaciones vale la regla general de la sección 4: si $r<r^*$ no hay ningún punto común; si $r=r^*$, lo común es un punto ($r=n$), una recta ($n-r=1$) o un plano ($n-r=2$). Los planos de un sistema homogéneo pasan por el origen, así que siempre se cortan, al menos, en $\mathbf 0$.

**Cómo leer la gráfica.** Cada ecuación tiene su propio estilo de línea (continua, discontinua, punteada...) además de su color, para que se distinga también en blanco y negro. El punto común es un círculo negro y la recta común, una línea negra gruesa. El título dice la configuración.

In [ ]:
def _paralelas(u, v):
    """True si los vectores renglón u y v (no nulos) son paralelos: uno es múltiplo del otro."""
    return sp.Matrix.vstack(u, v).rank() == 1


def _pares(indices):
    return [(indices[a], indices[c]) for a in range(len(indices)) for c in range(a + 1, len(indices))]


def geometria(Ab, explicar=False, n_cifras=4):
    """Configuración geométrica de un sistema de 2 incógnitas (rectas) o 3 (planos), decidida con los rangos.

    Devuelve un diccionario con n, m, r, r_aum, clave (texto corto: 'punto', 'misma recta', 'paralelas', 'recta',
    'mismo plano', 'prisma', 'dos paralelos y un tercero', 'paralelos', 'sin punto común', 'todo' o 'sin solución'),
    configuracion (la frase completa), notas, el punto común y la recta común (particular, dirección) si existen."""
    Ab = aumentada(Ab)
    m, n = Ab.rows, Ab.cols - 1
    if n not in (2, 3):
        raise ValueError(f"la interpretación geométrica es para sistemas de 2 o 3 incógnitas (este tiene {_incognitas(n)}); "
                         "para clasificarlo usa la sección 4")
    A, b = Ab[:, :n], Ab[:, n]
    el = _elimina(Ab, n, con_b=True)
    r, r_aum = el["r"], el["r_aum"]
    cero = sp.zeros(1, n)
    nulas = [i for i in range(m) if A.row(i) == cero]               # ecuaciones sin incógnitas: 0 = c
    figuras = [i for i in range(m) if i not in nulas]               # las que sí son rectas o planos
    k = len(figuras)
    objeto, objetos = ("recta", "rectas") if n == 2 else ("plano", "planos")
    notas = []
    for i in nulas:
        notas.append(f"la ecuación {i + 1} dice 0 = {b[i]}: " + ("ningún punto la cumple" if b[i] != 0 else
                     f"no es {'una recta' if n == 2 else 'un plano'} y no restringe nada"))
    punto = recta = None
    if r == r_aum:
        _, xp, dirs, _ = solucion_general(el["R"], el["pivotes"], n)
        if r == n:
            punto = xp
        elif n - r == 1:
            recta = (xp, dirs[0])
    iguales = [(i, j) for i, j in _pares(figuras) if sp.Matrix.vstack(Ab.row(i), Ab.row(j)).rank() == 1]
    paralelas = [(i, j) for i, j in _pares(figuras) if _paralelas(A.row(i), A.row(j)) and (i, j) not in iguales]
    if any(b[i] != 0 for i in nulas):
        clave, texto = "sin solución", f"sin solución: una ecuación pide 0 = c con c ≠ 0 (r = {r}, r* = {r_aum})"
    elif k == 0:
        clave, texto = "todo", f"todas las ecuaciones son 0 = 0: cualquier punto {'del plano' if n == 2 else 'del espacio'} es solución"
    elif n == 2 and k == 2:
        if r == 2:
            clave, texto = "punto", f"las rectas se cortan en un punto, {_tupla(punto)}"
        elif r_aum == 1:
            clave, texto = "misma recta", "son la misma recta: todos sus puntos son solución"
        else:
            clave, texto = "paralelas", "son paralelas y distintas: no hay punto común"
    elif n == 3 and k == 3:
        if r == 3:
            clave, texto = "punto", f"un punto común: los tres planos se cortan en {_tupla(punto)}"
        elif r == r_aum == 2:
            clave = "recta"
            if iguales:
                i, j = iguales[0]
                texto = f"una recta común: los planos {i + 1} y {j + 1} son el mismo y el tercero lo corta en una recta"
            else:
                texto = "una recta común: los tres planos pasan por una misma recta"
        elif r == r_aum == 1:
            clave, texto = "mismo plano", "los tres son el mismo plano: todos sus puntos son solución"
        elif r == 2:
            if not paralelas:
                clave = "prisma"
                texto = ("prisma: no hay punto común; los planos se cortan de dos en dos en tres rectas paralelas "
                         "y ningún par de normales es paralelo")
            else:
                i, j = paralelas[0]
                clave = "dos paralelos y un tercero"
                texto = (f"dos planos paralelos cortados por un tercero: no hay punto común "
                         f"(los planos {i + 1} y {j + 1} tienen normales paralelas)")
        else:
            clave = "paralelos"
            texto = "planos paralelos: no hay punto común" + (" (dos de ellos son el mismo plano)" if iguales else
                                                              " (los tres son distintos)")
    elif k == 1:
        clave, texto = f"mism{'a' if n == 2 else 'o'} {objeto}", f"una sola ecuación: todos los puntos de su {objeto} son solución"
    elif n == 3 and k == 2:
        if r == 2:
            clave, texto = "recta", "una recta común: los dos planos se cortan en una recta"
        elif r_aum == 1:
            clave, texto = "mismo plano", "son el mismo plano: todos sus puntos son solución"
        else:
            clave, texto = "paralelos", "planos paralelos y distintos: no hay punto común"
    else:                                                       # más ecuaciones que las de la tabla: regla general
        if r < r_aum:
            if r == 1:
                clave, texto = (f"paralel{'as' if n == 2 else 'os'}",
                                f"{objetos} paralel{'as' if n == 2 else 'os'}, no {'todas iguales' if n == 2 else 'todos iguales'}: no hay punto común")
            else:
                clave, texto = "sin punto común", f"no hay ningún punto común a {'las' if n == 2 else 'los'} {k} {objetos}"
        elif r == n:
            clave, texto = "punto", f"{'las' if n == 2 else 'los'} {k} {objetos} pasan por un mismo punto, {_tupla(punto)}"
        elif r == 1:
            clave = f"mism{'a' if n == 2 else 'o'} {objeto}"
            texto = f"{'todas son la misma recta' if n == 2 else 'todos son el mismo plano'}: todos sus puntos son solución"
        else:
            clave, texto = "recta", f"una recta común: los {k} planos pasan por una misma recta"
    res = {"n": n, "m": m, "r": r, "r_aum": r_aum, "clave": clave, "configuracion": texto, "notas": notas,
           "punto": punto, "recta": recta, "paralelas": paralelas, "iguales": iguales}
    if explicar:
        _informe_geometria(Ab, res, n_cifras)
    return res


def _informe_geometria(Ab, res, n_cifras):
    _imprime_sistema(Ab)
    print(f"r = rango(A) = {res['r']},  r* = rango[A | b] = {res['r_aum']},  n = {res['n']}")
    print("Configuración:", res["configuracion"] + ".")
    for nota in res["notas"]:
        print("Nota:", nota + ".")
    if res["punto"] is not None and not all(e.is_integer for e in res["punto"]):
        print("Punto común ≈ (" + ", ".join(cifras(e, n_cifras) for e in res["punto"]) + ")")
    if res["recta"] is not None:
        p, d = res["recta"]
        print(f"Recta común: {_tupla(p)} + s·{_tupla(d)}, con s cualquier número (s es la variable libre).")
    if res["n"] == 3 and res["clave"] in ("prisma", "dos paralelos y un tercero"):
        A = Ab[:, :3]
        figuras = [i for i in range(Ab.rows) if A.row(i) != sp.zeros(1, 3)]
        for i, j in _pares(figuras):
            cruz = A.row(i).T.cross(A.row(j).T)
            print(f"  n{_sub(i + 1)} × n{_sub(j + 1)} = {_tupla(cruz)}" + ("  → normales paralelas" if cruz == sp.zeros(3, 1) else ""))


def _caja(Ab, res, minimo):
    """Centro y medio ancho de la ventana: alrededor del punto o la recta común, o de los puntos de cada figura
    más cercanos al origen."""
    n = res["n"]
    A, b = _flotante(Ab[:, :n]), _flotante(Ab[:, n]).ravel()
    cercanos = [b[i] / (A[i] @ A[i]) * A[i] for i in range(A.shape[0]) if A[i] @ A[i] > 0]
    if res["punto"] is not None:
        centro = _flotante(res["punto"]).ravel()
    elif res["recta"] is not None:
        p, d = (_flotante(v).ravel() for v in res["recta"])
        centro = p - (p @ d) / (d @ d) * d                       # el punto de la recta más cercano al origen
    elif cercanos:
        centro = np.mean(cercanos, axis=0)
    else:
        centro = np.zeros(n)
    lejos = max([float(np.linalg.norm(q - centro)) for q in cercanos] + [0.0])
    return centro, float(math.ceil(max(minimo, 1.4 * lejos + 1.5)))


_DE_CANTO = ("prisma", "dos paralelos y un tercero")


def _direccion_comun(Ab):
    """Dirección común exacta de los planos (n_i × n_j de dos normales que no son paralelas), o None si no hay."""
    A = Ab[:, :3]
    filas = [i for i in range(Ab.rows) if A.row(i) != sp.zeros(1, 3)]
    for i, j in _pares(filas):
        d = A.row(i).T.cross(A.row(j).T)
        if d != sp.zeros(3, 1):
            return d
    return None


def _vista(Ab, res):
    """(elevación, giro) automáticos para 3D. En un prisma, o con dos planos paralelos y un tercero, se mira a lo largo
    de la dirección común de los planos (calculada exacta): se ven de canto, como un triángulo o como dos rectas
    paralelas cortadas por otra. En los demás casos se busca, cerca de la vista usual (20°, -60°), una en la que ningún
    plano quede de canto."""
    A = _flotante(Ab[:, :3])
    normales = [f / np.linalg.norm(f) for f in A if np.linalg.norm(f) > 0]
    if not normales:
        return 20, -60
    if res["clave"] in _DE_CANTO:
        d = _direccion_comun(Ab)
        if d is None:
            return 20, -60
        d = _flotante(d).ravel()
        d = -d if d[2] < 0 else d
        return math.degrees(math.atan2(d[2], math.hypot(d[0], d[1]))), math.degrees(math.atan2(d[1], d[0]))
    vistas = []
    for elev in range(15, 45, 5):
        for azim in range(-180, 180, 5):
            if min(azim % 90, 90 - azim % 90) < 20:           # casi a lo largo de un eje: sus marcas se enciman
                continue
            e, a = math.radians(elev), math.radians(azim)
            ojo = np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)])
            de_canto = min(abs(ojo @ nn) for nn in normales)       # 0: algún plano se ve como una recta
            vistas.append((de_canto, abs(elev - 20) + min(abs(azim + 60), 360 - abs(azim + 60)), elev, azim))
    buenas = [v for v in vistas if v[0] >= 0.25]                  # ningún plano a menos de unos 15° de verse de canto
    elegida = min(buenas, key=lambda v: v[1]) if buenas else max(vistas)
    return elegida[2], elegida[3]


def _seccion(Ab):
    """Para un prisma o dos planos paralelos con un tercero: el corte con el plano d·x = 0, perpendicular a la
    dirección común d. Devuelve (centro, medio ancho, trazas) con los vértices de ese corte calculados exactos y la
    traza de cada plano (renglón, punto, dirección unitaria), que es la recta que se ve al mirar a lo largo de d."""
    A, b = Ab[:, :3], Ab[:, 3]
    d = _direccion_comun(Ab)
    filas = [i for i in range(Ab.rows) if A.row(i) != sp.zeros(1, 3)]
    vertices = []
    for i, j in _pares(filas):
        if A.row(i).T.cross(A.row(j).T) != sp.zeros(3, 1):        # dos planos que se cortan: su recta pasa por el corte
            M = sp.Matrix.vstack(A.row(i), A.row(j), d.T)
            vertices.append(_flotante(M.solve(sp.Matrix([b[i], b[j], 0]))).ravel())
    V = np.array(vertices)
    centro = V.mean(axis=0)
    L = max(1.0, 1.8 * float(np.max(np.linalg.norm(V - centro, axis=1))))
    dn = _flotante(d).ravel() / float(np.linalg.norm(_flotante(d)))
    trazas = []
    for i in filas:
        n_i = _flotante(A.row(i)).ravel()
        punto = centro + (float(b[i]) - n_i @ centro) / (n_i @ n_i) * n_i
        u = np.cross(dn, n_i)
        trazas.append((i, punto, u / np.linalg.norm(u)))
    return centro, L, trazas


def _dentro(p, u, bajo, alto):
    """Intervalo de t en el que p + t·u queda dentro de la caja [bajo, alto]."""
    t0, t1 = -np.inf, np.inf
    for k in range(len(p)):
        if abs(u[k]) > 1e-12:
            a, c = sorted(((bajo[k] - p[k]) / u[k], (alto[k] - p[k]) / u[k]))
            t0, t1 = max(t0, a), min(t1, c)
    return t0, t1


def graficar(Ab, giro=None, elevacion=None, n_cifras=4):
    """Dibuja las rectas (2 incógnitas) o los planos (3 incógnitas) del sistema, marca lo que tienen en común
    y escribe la configuración en el título. En 3D, sin giro ni elevación, la vista se elige sola (_vista); en un
    prisma o con dos planos paralelos y un tercero, cada plano se ve de canto y se dibuja como una recta gruesa
    con su estilo y su número."""
    Ab = aumentada(Ab)
    res = geometria(Ab)
    n, m = res["n"], Ab.rows
    A, b = _flotante(Ab[:, :n]), _flotante(Ab[:, n]).ravel()
    if not any(np.any(A[i] != 0) for i in range(m)):
        print("No hay nada que dibujar: todas las ecuaciones tienen sus coeficientes en cero, así que ninguna es "
              f"{'una recta' if n == 2 else 'un plano'}.")
        return res
    nombres = nombres_incognitas(n)
    centro, L = _caja(Ab, res, 4 if n == 2 else 3)
    titulo = f"r = {res['r']}, r* = {res['r_aum']}: {res['configuracion']}"
    if n == 2:
        fig, ax = plt.subplots(figsize=(6.2, 6.2))
        for i in range(m):
            a1, a2 = A[i]
            if a1 == 0 and a2 == 0:
                continue
            if abs(a2) >= abs(a1):
                xs = np.linspace(centro[0] - L, centro[0] + L, 2)
                ys = (b[i] - a1 * xs) / a2
            else:
                ys = np.linspace(centro[1] - L, centro[1] + L, 2)
                xs = (b[i] - a2 * ys) / a1
            ax.plot(xs, ys, linestyle=_ESTILOS[i], color=_COLORES[i], lw=2.2,
                    label=f"ecuación {i + 1}: {_ecuacion(Ab.row(i), nombres)}")
        if res["punto"] is not None:
            p = _flotante(res["punto"]).ravel()
            ax.plot(*p, "o", color="black", ms=9, zorder=5, label=f"punto común {_tupla(res['punto'])}")
        ax.set_xlim(centro[0] - L, centro[0] + L); ax.set_ylim(centro[1] - L, centro[1] + L); ax.set_aspect("equal")
        ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
        ax.grid(alpha=0.3); ax.set_xlabel("x"); ax.set_ylabel("y")
        lugar = "best"
    else:
        fig = plt.figure(figsize=(7.5, 6.8))
        ax = fig.add_subplot(projection="3d")
        de_canto = giro is None and elevacion is None and res["clave"] in _DE_CANTO and _direccion_comun(Ab) is not None
        if de_canto:                                             # cada plano se ve como una recta: una línea gruesa por plano
            centro, L, trazas = _seccion(Ab)
            bajo, alto = centro - L, centro + L
            for i, p, u in trazas:
                t0, t1 = _dentro(p, u, bajo, alto)
                P = p[None, :] + np.array([t0, t1])[:, None] * u[None, :]
                ax.plot(*P.T, linestyle=_ESTILOS[i], color=_COLORES[i], lw=3.5,
                        label=f"plano {i + 1}: {_ecuacion(Ab.row(i), nombres)}")
                ax.text(*(p + 0.92 * t1 * u), f" {i + 1}", fontsize=13, fontweight="bold", color="black")
            titulo += ". Vista a lo largo de la dirección común: cada plano se ve de canto, como una recta con su número"
        else:
            bajo, alto = centro - L, centro + L
            for i in range(m):
                normal = A[i]
                if not normal.any():
                    continue
                k = int(np.argmax(np.abs(normal)))             # se despeja la variable de mayor coeficiente
                u, v = [j for j in range(3) if j != k]
                U, V = np.meshgrid(np.linspace(bajo[u], alto[u], 9), np.linspace(bajo[v], alto[v], 9))
                W = (b[i] - normal[u] * U - normal[v] * V) / normal[k]
                W = np.where((W < bajo[k] - 1e-9) | (W > alto[k] + 1e-9), np.nan, W)   # fuera de la caja no se dibuja
                XYZ = [None, None, None]
                XYZ[u], XYZ[v], XYZ[k] = U, V, W
                ax.plot_surface(*XYZ, color=_COLORES[i], alpha=0.08, linewidth=0, shade=False)
                ax.plot_wireframe(*XYZ, color=_COLORES[i], linestyle=_ESTILOS[i], linewidth=1.2,
                                  label=f"plano {i + 1}: {_ecuacion(Ab.row(i), nombres)}")
            if res["punto"] is not None:
                p = _flotante(res["punto"]).ravel()
                ax.scatter(*p, s=90, color="black", depthshade=False, label=f"punto común {_tupla(res['punto'])}")
            if res["recta"] is not None:
                p, d = (_flotante(w).ravel() for w in res["recta"])
                t0, t1 = _dentro(p, d / np.linalg.norm(d), bajo, alto)
                if t0 < t1:
                    P = p[None, :] + np.array([t0, t1])[:, None] * (d / np.linalg.norm(d))[None, :]
                    ax.plot(*P.T, color="black", lw=3.5, label="recta común")
        ax.set_xlim(bajo[0], alto[0]); ax.set_ylim(bajo[1], alto[1]); ax.set_zlim(bajo[2], alto[2])
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.set_box_aspect((1, 1, 1))                         # misma escala en los tres ejes: los ángulos se ven bien
        ax.set_proj_type("ortho")                            # sin perspectiva, un plano de canto se ve como una recta
        auto = _vista(Ab, res)
        ax.view_init(elev=auto[0] if elevacion is None else elevacion, azim=auto[1] if giro is None else giro)
        lugar = "upper left"
    ax.set_title(textwrap.fill(titulo, 62), fontsize=10)
    if ax.get_legend_handles_labels()[0]:                    # solo si hay algo rotulado
        ax.legend(loc=lugar, fontsize=8)
    plt.show()
    return res

**Recorrido paso a paso.** Primero, tres planos con un solo punto común: $x+y+z=3$, $x-y+2z=2$, $2x+y-z=2$. Después, $x+y=2$, $y+z=2$, $x-z=1$: el primer renglón menos el segundo da $x-z=0$, que choca con la tercera ecuación ($x-z=1$), así que $r=2$ y $r^*=3$; ningún par de normales es paralelo, de modo que forman un prisma. La vista automática mira a lo largo de la dirección común de los tres planos, $\mathbf n_1\times\mathbf n_2$: los planos se ven de canto y cada uno se dibuja como una recta gruesa con su estilo y su número, y el prisma aparece como un triángulo. En la graficadora puedes desmarcar la vista automática y girarlo.

In [ ]:
geometria("1, 1, 1, 3; 1, -1, 2, 2; 2, 1, -1, 2", explicar=True)
graficar("1, 1, 1, 3; 1, -1, 2, 2; 2, 1, -1, 2")
print()
geometria("1, 1, 0, 2; 0, 1, 1, 2; 1, 0, -1, 1", explicar=True)
graficar("1, 1, 0, 2; 0, 1, 1, 2; 1, 0, -1, 1");

**Graficadora.** Escribe un sistema de 2 incógnitas (rectas) o de 3 (planos) como matriz aumentada. Prueba también los de 3 incógnitas de los casos de prueba de abajo, por ejemplo `1, 0, 1, 2; 0, 1, 1, 3; 1, 1, 2, 5` (una recta común) o `1, 1, 1, 1; 1, 1, 1, 2; 1, 1, 1, 3` (planos paralelos). En 3D la vista se elige sola: en general, de modo que ningún plano quede de canto; en un prisma o con dos planos paralelos cortados por un tercero, a lo largo de la dirección común de los planos, que entonces se ven de canto, como rectas numeradas 1, 2, 3. Desmarca «vista automática» para girarla con los controles.

In [ ]:
def calculadora_geometria(Ab_txt, vista_automatica, giro, elevacion, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; la gráfica va después
        try:
            Ab = aumentada(Ab_txt)
            res = geometria(Ab)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_geometria(Ab, res, n_cifras)
    if vista_automatica:
        graficar(Ab)
    else:
        graficar(Ab, giro=giro, elevacion=elevacion)


widgets.interact(calculadora_geometria,
    Ab_txt=widgets.Textarea(value="2, 1, 5\n1, 3, 5", description="[A | b] =", continuous_update=False,
                            layout=widgets.Layout(width="340px", height="110px")),
    vista_automatica=widgets.Checkbox(value=True, description="vista automática (3D)"),
    giro=widgets.IntSlider(value=-60, min=-180, max=180, step=15, description="giro", continuous_update=False),
    elevacion=widgets.IntSlider(value=20, min=-10, max=90, step=5, description="elevación", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("2, 1, 5; 1, 3, 5 → las rectas se cortan en (2, 1)",
     lambda: (lambda g: g["clave"] == "punto" and g["punto"] == vector("2, 1")
                        and g["configuracion"].startswith("las rectas se cortan en un punto"))(geometria("2, 1, 5; 1, 3, 5"))),
    ("1, -2, 1; -2, 4, 3 → paralelas y distintas (r = 1, r* = 2)",
     lambda: (lambda g: g["clave"] == "paralelas" and (g["r"], g["r_aum"]) == (1, 2)
                        and "son paralelas y distintas" in g["configuracion"])(geometria("1, -2, 1; -2, 4, 3"))),
    ("1, -1, 1; 3, -3, 3 → la misma recta (r = r* = 1)",
     lambda: (lambda g: g["clave"] == "misma recta" and "son la misma recta" in g["configuracion"])(geometria("1, -1, 1; 3, -3, 3"))),
    ("1, 1, 1, 3; 1, -1, 2, 2; 2, 1, -1, 2 → un punto común, (1, 1, 1)",
     lambda: (lambda g: g["clave"] == "punto" and g["punto"] == vector("1, 1, 1")
                        and g["configuracion"].startswith("un punto común"))(geometria("1, 1, 1, 3; 1, -1, 2, 2; 2, 1, -1, 2"))),
    ("1, 0, 1, 2; 0, 1, 1, 3; 1, 1, 2, 5 → una recta común, que está en los tres planos",
     lambda: (lambda g: g["clave"] == "recta" and g["configuracion"].startswith("una recta común")
                        and all(matriz("1, 0, 1; 0, 1, 1; 1, 1, 2") * (g["recta"][0] + t * g["recta"][1]) == vector("2, 3, 5")
                                for t in (0, 1, -3)))(geometria("1, 0, 1, 2; 0, 1, 1, 3; 1, 1, 2, 5"))),
    ("1, 1, 0, 2; 0, 1, 1, 2; 1, 0, -1, 1 → prisma; y 1, 1, 1, 1; 1, 1, 1, 2; 1, 1, 1, 3 → planos paralelos",
     lambda: (lambda g1, g2: g1["clave"] == "prisma" and g1["configuracion"].startswith("prisma")
                             and g2["clave"] == "paralelos" and g2["configuracion"].startswith("planos paralelos"))(
         geometria("1, 1, 0, 2; 0, 1, 1, 2; 1, 0, -1, 1"), geometria("1, 1, 1, 1; 1, 1, 1, 2; 1, 1, 1, 3"))),
]
corre_pruebas(PRUEBAS_5, 5)

**Contrasta (sección 5).** Antes de graficar $x+y=2$, $2x+2y=5$, predice con los rangos qué vas a ver: calcula $r$ y $r^*$ y escribe una de estas tres frases: «se cortan en un punto», «son la misma recta» o «son paralelas y distintas».

In [ ]:
mi_r = None              # tu rango de A, por ejemplo 2
mi_r_aum = None          # tu rango de [A | b], por ejemplo 1
mi_prediccion = None     # una de estas frases: "se cortan en un punto", "son la misma recta" o "son paralelas y distintas"

# Frases que se aceptan, ya normalizadas. Se compara la frase completa, así que una negación («no se cortan»)
# no cuenta como ninguna de las tres.
_PREDICCIONES = {
    "punto": ["se cortan en un punto", "se cortan", "se cortan en un solo punto", "un punto"],
    "misma recta": ["son la misma recta", "la misma recta", "misma recta"],
    "paralelas": ["son paralelas y distintas", "paralelas y distintas", "son paralelas", "paralelas"],
}


def _prediccion(texto):
    t = normaliza_texto(texto)
    for clave, frases in _PREDICCIONES.items():
        if t in frases:
            return clave
    raise ValueError(f"«{texto}» no es una de las tres frases: escribe «se cortan en un punto», «son la misma recta» "
                     "o «son paralelas y distintas»")


if mi_r is None or mi_r_aum is None or mi_prediccion is None:     # la referencia se muestra después de tu predicción
    print("falta tu predicción: escribe r, r* y lo que esperas ver, y vuelve a ejecutar la celda")
else:
    try:
        mio_r, mio_r_aum = numero(mi_r, "tu r"), numero(mi_r_aum, "tu r*")
        mia = _prediccion(mi_prediccion)
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        if not all(v.is_integer and 0 <= v <= 2 for v in (mio_r, mio_r_aum)):   # respuesta imposible: sin referencia
            print("Con 2 ecuaciones y 2 incógnitas, r y r* son enteros de 0 a 2: revísalos y vuelve a ejecutar la celda.")
        else:
            ref = geometria("1, 1, 2; 2, 2, 5")
            print(f"La calculadora da: r = {ref['r']}, r* = {ref['r_aum']}; {ref['configuracion']}.")
            print("Rangos:", "coinciden" if (mio_r, mio_r_aum) == (ref["r"], ref["r_aum"]) else
                  "NO coinciden: escalona [A | b]; si el renglón 2 de A queda en ceros, mira qué queda en la columna de b.")
            print("Predicción:", "coincide" if mia == ref["clave"] else
                  "NO coincide: con r = 1 las rectas tienen la misma dirección; r* dice si además son la misma recta.")
            graficar("1, 1, 2; 2, 2, 5")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
fallas, total = [], 0
for k, pruebas in enumerate([PRUEBAS_1, PRUEBAS_2, PRUEBAS_3, PRUEBAS_4, PRUEBAS_5], 1):
    resultados = evalua(pruebas)                      # cada caso se ejecuta una sola vez
    print(f"Sección {k}: {sum(ok for _, ok in resultados)}/{len(pruebas)}")
    fallas += [f"sección {k}: {nombre}" for nombre, ok in resultados if not ok]
    total += len(pruebas)
for falla in fallas:
    print("FALLA", falla)
assert not fallas, f"Fallan {len(fallas)} de {total} pruebas (están arriba)"
print(f"Todas las verificaciones pasaron: {total}/{total}")